In [2]:
import numpy as np
import pandas as pd
import random
import math
import time
from collections import deque

In [3]:
ratings_col = ['User ID', 'Movie ID', 'Rating', 'Timestamp']
movies_col = ['Movie ID', 'Title', 'Genres']
users_col = ['User ID', 'Gender', 'Age', 'Occupation', 'Zip-Code']

ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", names=ratings_col, engine="python")
movies = pd.read_csv("ml-1m/movies.dat", sep="::", names=movies_col, engine="python", encoding="latin-1")
users = pd.read_csv("ml-1m/users.dat", sep="::", names=users_col, engine="python")


In [4]:
ratings_users = pd.merge(ratings, users, on = 'User ID')
df1 = pd.merge(ratings_users, movies, on = 'Movie ID')

In [5]:
df1.head()

,User ID,Movie ID,Rating,Timestamp,Gender,Age,Occupation,Zip-Code,Title,Genres
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy


"""
FL device selection (MovieLens) + DQN agent for device selection.
- Loads MovieLens 1M from local file path.
- Preprocess top-1000 movies -> binary user×movie matrix.
- Split users into devices: [1000,900,...,100] (sum=5500).
- Build device stats and train a small DQN to select devices.
- After selection, greedy offloader assigns data from non-selected -> selected.
"""

from sklearn.metrics.pairwise import cosine_similarity

# PyTorch for DQN
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# -------------------------
# Configuration / constants
# -------------------------
MOVIELENS_PATH = r'C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m'  # change if needed

# Problem constants (you can tune these)
MODEL_SIZE = 5e6      # bits (example)
BANDWIDTH_DN = 50e6   # bps
BANDWIDTH_UP = 10e6   # bps
P_S = 0.5
P_N = 0.3
EPOCHS = 1

# Constraints — adjust if infeasible
CONSTRAINTS = {
    'z_min': 50,
    'z_max': 5000,
    'D_max': 700,
    't_max': 0.1,
    'e_max': 0.5,
    'S_min': 2,
    'S_max': 9,
    'alpha_n': 0.1
}

# Device sizes (sum should be <= number of users you keep)
df = df1.sort_values('Timestamp')
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

N = 10     # No of edge devices
unique_users = train['User ID'].unique()                # unique users in the training set
np.random.shuffle(unique_users)

real_user_groups = []
start = 0
user_groups = np.random.multinomial(len(unique_users), [1/N]*N)          # split users into N groups
for size in user_groups:
    real_user_groups.append(unique_users[start:start+size])
    start += size

# Now you can do:
group_sizes = [len(g) for g in real_user_groups]

# DQN hyperparameters
EPISODES = 400
BATCH_SIZE = 64
LR = 1e-3
GAMMA = 0.99
EPS_START = 1.0
EPS_MIN = 0.05
EPS_DECAY = 0.995
REPLAY_CAPACITY = 20000

# -------------------------
# Utilities
# -------------------------
def log2(x):
    return math.log(x, 2.0)

def cosine_sim_vecs(a, b):
    na = np.linalg.norm(a) + 1e-9
    nb = np.linalg.norm(b) + 1e-9
    return float(np.dot(a, b) / (na * nb))

# -------------------------
# MovieLens loader + preprocessing
# -------------------------
def load_movielens_binary_matrix(path_dir, top_k_movies=1000):
    """
    Load MovieLens 1M from path_dir (folder containing ratings.dat, movies.dat, users.dat)
    Returns: request_matrix (pandas DataFrame) with UserID as index, MovieID columns, values {0,1}
    """
    ratings_path = path_dir.rstrip('/') + '/ratings.dat'
    movies_path = path_dir.rstrip('/') + '/movies.dat'
    users_path = path_dir.rstrip('/') + '/users.dat'
    print("Loading ratings from:", ratings_path)
    ratings_col = ['UserID', 'MovieID', 'Rating', 'Timestamp']
    ratings = pd.read_csv(ratings_path, sep='::', names=ratings_col, engine='python')
    # Filter top-k movies
    top_movies = ratings['MovieID'].value_counts().nlargest(top_k_movies).index
    ratings = ratings[ratings['MovieID'].isin(top_movies)]
    # Pivot -> binary request matrix
    request_matrix = ratings.pivot_table(index='UserID', columns='MovieID', values='Rating', fill_value=0)
    request_matrix = (request_matrix > 0).astype(int)
    print("Request matrix shape (users x movies):", request_matrix.shape)
    return request_matrix

# -------------------------
# Build devices from request matrix
# -------------------------
def create_devices_from_matrix(request_matrix, device_sizes, seed=0):
    rng = np.random.default_rng(seed)
    user_ids = request_matrix.index.tolist()
    rng.shuffle(user_ids)
    devices = []
    start = 0
    for i, size in enumerate(device_sizes):
        users = user_ids[start:start+size]
        start += size
        submat = request_matrix.loc[users].values  # users x items (numpy)
        D_orig = len(users)
        z_orig = int(submat.sum())
        # Compute rho: average pairwise cosine similarity across users (sample if too many)
        if D_orig <= 1:
            rho = 1.0
        else:
            Xf = submat.astype(float)
            # sample at most 50 rows for speed
            if D_orig > 50:
                idxs = np.random.choice(D_orig, size=50, replace=False)
                Xs = Xf[idxs]
            else:
                Xs = Xf
            sims = cosine_similarity(Xs)
            # zero diagonal
            np.fill_diagonal(sims, 0.0)
            total_pairs = Xs.shape[0] * (Xs.shape[0] - 1)
            rho = float(sims.sum() / total_pairs) if total_pairs > 0 else 0.0
        # hetero params (randomized plausible values)
        f_n = float(np.random.uniform(1.0, 2.0) * 1e9)
        beta_n = float(np.random.uniform(1e-28, 1e-27))
        C_n = int(np.random.randint(5, 10))
        gamma_n = float(np.random.uniform(10, 20))
        # precompute times/energies
        t_dn = MODEL_SIZE / (BANDWIDTH_DN * log2(1 + gamma_n))
        # choose an upload model size wn (bits) small relative to model
        wn = 2e6
        t_up = wn / (BANDWIDTH_UP * log2(1 + gamma_n))
        t_comp_coef = (EPOCHS * C_n) / f_n
        e_dn = P_S * t_dn
        e_up = P_N * t_up
        e_comp_coef = beta_n * EPOCHS * C_n * (f_n ** 2)
        devices.append({
            'id': i,
            'users': users,
            'D_orig': D_orig,
            'z_orig': z_orig,
            'rho': rho,
            'f_n': f_n,
            'beta_n': beta_n,
            'C_n': C_n,
            'gamma_n': gamma_n,
            't_dn': t_dn,
            't_up': t_up,
            't_comp_coef': t_comp_coef,
            'e_dn': e_dn,
            'e_up': e_up,
            'e_comp_coef': e_comp_coef,
            'wn': wn
        })
    return devices

# -------------------------
# Feasibility helpers + greedy offloader
# -------------------------
def device_latency(dev, D):
    return dev['t_dn'] + dev['t_comp_coef'] * D + dev['t_up']

def device_energy(dev, D):
    return dev['e_dn'] + dev['e_comp_coef'] * D + dev['e_up']

def feasible_selected_device(dev, D_val, z_val):
    if D_val > CONSTRAINTS['D_max']:
        return False
    if z_val < CONSTRAINTS['z_min']:
        return False
    if device_energy(dev, D_val) > CONSTRAINTS['e_max']:
        return False
    return True

def feasible_round_latency(selected_ids, D_vals, devices):
    if not selected_ids:
        return False
    latencies = [device_latency(devices[i], D_vals[i]) for i in selected_ids]
    T = max(latencies) if latencies else 0.0
    return T <= CONSTRAINTS['t_max']

def non_selected_request_ok(z_val, T):
    return (z_val + CONSTRAINTS['alpha_n'] * T) <= CONSTRAINTS['z_max']

def greedy_offloading(devices, selected_idx, D_vals, z_vals):
    # identical approach to prior greedy_offloading: offload chunks from non-selected -> selected
    if not selected_idx:
        return
    N = len(devices)
    per_unit_z = np.zeros(N)
    for i, d in enumerate(devices):
        per_unit_z[i] = d['z_orig'] / max(1.0, d['D_orig'])
    selset = set(selected_idx)
    non_selected = [i for i in range(N) if i not in selset]
    for src in non_selected:
        remaining = float(devices[src]['D_orig'])
        if remaining <= 0 or per_unit_z[src] <= 0:
            continue
        CHUNK = max(1.0, devices[src]['D_orig'] / 20.0)
        while remaining > 1e-9:
            best_gain = 0.0
            best_j = None
            best_chunk = 0.0
            for j in selected_idx:
                if j == src:
                    continue
                cap = CONSTRAINTS['D_max'] - D_vals[j]
                if cap <= 0:
                    continue
                chunk = min(CHUNK, remaining, cap)
                if chunk <= 0:
                    continue
                D_new_j = D_vals[j] + chunk
                z_new_j = z_vals[j] + per_unit_z[src] * chunk
                # per-selected check
                if not feasible_selected_device(devices[j], D_new_j, z_new_j):
                    continue
                # round latency check
                D_temp = D_vals.copy()
                D_temp[j] = D_new_j
                if not feasible_round_latency(selected_idx, D_temp, devices):
                    continue
                gain = devices[j]['rho'] * (per_unit_z[src] * chunk)
                if gain > best_gain:
                    best_gain = gain
                    best_j = j
                    best_chunk = chunk
            if best_j is None:
                if CHUNK > 1.0:
                    CHUNK = max(1.0, CHUNK / 2.0)
                    continue
                else:
                    break
            # commit
            D_vals[best_j] += best_chunk
            z_vals[best_j] += per_unit_z[src] * best_chunk
            D_vals[src] = max(0.0, D_vals[src] - best_chunk)
            z_vals[src] = max(0.0, z_vals[src] - per_unit_z[src] * best_chunk)
            remaining -= best_chunk

# -------------------------
# Gym-like environment (device selection)
# -------------------------
class FederatedEnv:
    def __init__(self, devices):
        self.devices = devices
        self.N = len(devices)
        self.S_max = CONSTRAINTS['S_max']
        self.S_min = CONSTRAINTS['S_min']
        self.reset()

    def reset(self):
        self.D = np.array([d['D_orig'] for d in self.devices], dtype=float)
        self.z = np.array([d['z_orig'] for d in self.devices], dtype=float)
        self.selected = []
        self.done = False
        return self._build_obs()

    def _build_obs(self):
        obs = []
        for i, d in enumerate(self.devices):
            Dn = self.D[i] / max(1.0, CONSTRAINTS['D_max'])
            zn = self.z[i] / max(1.0, CONSTRAINTS['z_max'])
            rho = d['rho']
            en = device_energy(d, self.D[i]) / max(1e-9, CONSTRAINTS['e_max'])
            lat = device_latency(d, self.D[i]) / max(1e-9, CONSTRAINTS['t_max'])
            sel = 1.0 if i in self.selected else 0.0
            obs.extend([Dn, zn, rho, en, lat, sel])
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("step after done")
        reward = 0.0
        info = {}
        # invalid or repeated picks -> small negative, end episode
        if action in self.selected:
            self.done = True
            return self._build_obs(), -1.0, True, info
        self.selected.append(int(action))
        if len(self.selected) > self.S_max:
            self.done = True
            return self._build_obs(), -5.0, True, info
        if len(self.selected) < self.S_max:
            return self._build_obs(), 0.0, False, info
        # episode termination: evaluate selection
        self.done = True
        # precheck
        if not all(feasible_selected_device(self.devices[i], self.D[i], self.z[i]) for i in self.selected):
            return self._build_obs(), -50.0, True, info
        if not feasible_round_latency(self.selected, self.D, self.devices):
            return self._build_obs(), -50.0, True, info
        # greedy offload
        D_vals = self.D.copy()
        z_vals = self.z.copy()
        greedy_offloading(self.devices, self.selected, D_vals, z_vals)
        T = max(device_latency(self.devices[i], D_vals[i]) for i in self.selected) if self.selected else 0.0
        # non-selected req check
        if not all(non_selected_request_ok(z_vals[i], T) for i in range(self.N) if i not in self.selected):
            return self._build_obs(), -50.0, True, info
        if not all(feasible_selected_device(self.devices[i], D_vals[i], z_vals[i]) for i in self.selected):
            return self._build_obs(), -50.0, True, info
        if T > CONSTRAINTS['t_max']:
            return self._build_obs(), -50.0, True, info
        obj = float(sum(z_vals[i] * self.devices[i]['rho'] for i in self.selected))
        reward = obj / 1e6  # scaled
        info['objective'] = obj
        info['T'] = T
        info['D_vals'] = D_vals
        info['z_vals'] = z_vals
        return self._build_obs(), reward, True, info

# -------------------------
# DQN (if torch available)
# -------------------------
if TORCH_OK:
    class ReplayBuffer:
        def __init__(self, capacity=REPLAY_CAPACITY):
            self.buf = deque(maxlen=capacity)
        def push(self, s,a,r,s2,d):
            self.buf.append((s,a,r,s2,d))
        def sample(self, batch_size):
            batch = random.sample(self.buf, batch_size)
            s,a,r,s2,d = map(np.array, zip(*batch))
            return s,a,r,s2,d
        def __len__(self):
            return len(self.buf)

    class QNet(nn.Module):
        def __init__(self, in_dim, out_dim, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim)
            )
        def forward(self, x):
            return self.net(x)

    class DQNAgent:
        def __init__(self, obs_dim, n_actions, lr=LR):
            self.q = QNet(obs_dim, n_actions)
            self.target = QNet(obs_dim, n_actions)
            self.target.load_state_dict(self.q.state_dict())
            self.opt = optim.Adam(self.q.parameters(), lr=lr)
            self.replay = ReplayBuffer()
            self.eps = EPS_START
            self.n_actions = n_actions
            self.step_count = 0

        def select(self, state):
            if random.random() < self.eps:
                return random.randrange(self.n_actions)
            s = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                qvals = self.q(s).cpu().numpy()[0]
            return int(np.argmax(qvals))

        def store(self, s,a,r,s2,d):
            self.replay.push(s,a,r,s2,d)

        def update(self, batch_size=BATCH_SIZE):
            if len(self.replay) < batch_size:
                return
            s,a,r,s2,d = self.replay.sample(batch_size)
            s = torch.FloatTensor(s)
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r)
            s2 = torch.FloatTensor(s2)
            d = torch.FloatTensor(d)
            qvals = self.q(s).gather(1, a).squeeze()
            with torch.no_grad():
                qnext = self.target(s2).max(1)[0]
                target = r + (1-d) * GAMMA * qnext
            loss = nn.MSELoss()(qvals, target)
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
            self.step_count += 1
            self.eps = max(EPS_MIN, self.eps * EPS_DECAY)
            if self.step_count % 500 == 0:
                self.target.load_state_dict(self.q.state_dict())

# -------------------------
# Training loop
# -------------------------
def train_dqn_on_movielens(devices, episodes=EPISODES):
    env = FederatedEnv(devices)
    obs0 = env.reset()
    obs_dim = obs0.shape[0]
    n_actions = len(devices)
    if not TORCH_OK:
        raise RuntimeError("PyTorch not installed. Install torch to train DQN.")
    agent = DQNAgent(obs_dim, n_actions, lr=LR)
    best = {'obj': -1.0, 'info': None, 'selected': None}
    rewards = []
    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action = agent.select(state)
            next_s, r, done, info = env.step(action)
            agent.store(state, action, r, next_s, float(done))
            agent.update(BATCH_SIZE)
            state = next_s
            ep_reward += r
        rewards.append(ep_reward)
        if info and 'objective' in info:
            obj = info['objective']
            if obj > best['obj']:
                best['obj'] = obj
                best['info'] = info
                best['selected'] = env.selected.copy()
        if (ep+1) % 50 == 0:
            print(f"Episode {ep+1}/{episodes} — ep_reward={ep_reward:.3f} eps={agent.eps:.3f} best_obj={best['obj']:.1f}")
    return agent, best, rewards

def get_random_device_sizes(num_users, num_devices=10, seed=0):
    """
    Randomly split num_users into num_devices groups.
    Returns a list of device sizes (summing exactly to num_users).
    """
    rng = np.random.default_rng(seed)
    probs = np.ones(num_devices) / num_devices
    sizes = rng.multinomial(num_users, probs)
    return sizes.tolist()

def print_solution_details(best, devices):
    if best['info'] is None:
        print("No feasible solution found.")
        return
    
    selected = best['selected']
    D_vals = best['info']['D_vals']
    z_vals = best['info']['z_vals']
    T = best['info']['T']
    
    print("\n=== DETAILED SOLUTION STATS ===")
    print(f"Selected devices ({len(selected)}): {selected}")
    print(f"Objective: {best['info']['objective']:.2f}")
    print(f"Round latency T: {T:.4f} s")
    
    print("\nPer-device breakdown:")
    for i, dev in enumerate(devices):
        sel = "✅" if i in selected else "❌"
        D = D_vals[i]
        z = z_vals[i]
        rho = dev['rho']
        lat = device_latency(dev, D)
        ene = device_energy(dev, D)
        print(f" Device {i:2d} {sel} | Users={dev['D_orig']} | D={D:.1f} | z={z:.1f} | rho={rho:.3f} "
              f"| Lat={lat:.4f} s | E={ene:.3f} J")

# -------------------------
# Main
# -------------------------
def main():
    print("Loading MovieLens top-1000 movies and creating devices...")
    request_matrix = load_movielens_binary_matrix(MOVIELENS_PATH, top_k_movies=1000)

    num_users = request_matrix.shape[0]   # e.g. 6040 in ML-1M top-1000
    num_devices = 10                      # can change this
    device_sizes = get_random_device_sizes(num_users, num_devices, seed=42)

    print("Random device sizes:", device_sizes, " (sum =", sum(device_sizes), ")")

    devices = create_devices_from_matrix(request_matrix, device_sizes, seed=1)
    print("Devices prepared. D_orig list:", [d['D_orig'] for d in devices])

    if not TORCH_OK:
        print("PyTorch not available. Install torch and re-run to train DQN.")
        return

    agent, best, rewards = train_dqn_on_movielens(devices, episodes=EPISODES)

    print("\n=== BEST SOLUTION FOUND ===")
    if best['info'] is not None:
        print_solution_details(best, devices)
    else:
        print("No feasible solution found.")



if __name__ == "__main__":
    main()


"""
FL device selection (MovieLens) + improved DQN agent for device selection.
- STOP action added (so agent chooses number of devices).
- Softer greedy offloading (partial chunks).
- Immediate marginal reward on selection.
- Prints detailed solution stats at the end.
"""

import math
import random
import time
from collections import deque

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# PyTorch for DQN
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# -------------------------
# Configuration / constants
# -------------------------
MOVIELENS_PATH = r'C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m'  # change if needed

MODEL_SIZE = 5e6      # bits (example)
BANDWIDTH_DN = 50e6   # bps
BANDWIDTH_UP = 10e6   # bps
P_S = 0.5
P_N = 0.3
EPOCHS = 1
# HYPERPARAMETERS DEFINED

CONSTRAINTS = {
    'z_min': 50,
    'z_max': 5000,
    'D_max': 2000,
    't_max': 0.1,
    'e_max': 0.5,
    'S_min': 2,
    'S_max': 9,
    'alpha_n': 0.1
}
# CONSTRAINTS DEFINED

# DQN hyperparameters
EPISODES = 400
BATCH_SIZE = 64
LR = 1e-3
GAMMA = 0.99
EPS_START = 1.0
EPS_MIN = 0.05
EPS_DECAY = 0.995
REPLAY_CAPACITY = 20000
SEED = 1

np.random.seed(SEED)
random.seed(SEED)
if TORCH_OK:
    torch.manual_seed(SEED)

# -------------------------
# Utilities
# -------------------------
def log2(x):
    return math.log(x, 2.0)

# -------------------------
# MovieLens loader + preprocessing
# -------------------------
def load_movielens_binary_matrix(path_dir, top_k_movies=1000):
    ratings_path = path_dir.rstrip('/') + '/ratings.dat'
    ratings_col = ['UserID', 'MovieID', 'Rating', 'Timestamp']
    print("Loading ratings from:", ratings_path)
    ratings = pd.read_csv(ratings_path, sep='::', names=ratings_col, engine='python')
    top_movies = ratings['MovieID'].value_counts().nlargest(top_k_movies).index
    ratings = ratings[ratings['MovieID'].isin(top_movies)]
    request_matrix = ratings.pivot_table(index='UserID', columns='MovieID', values='Rating', fill_value=0)
    request_matrix = (request_matrix > 0).astype(int)
    print("Request matrix shape (users x movies):", request_matrix.shape)
    return request_matrix

# -------------------------
# Build devices from request matrix
# -------------------------
def create_devices_from_matrix(request_matrix, device_sizes, seed=0):
    rng = np.random.default_rng(seed)
    user_ids = request_matrix.index.tolist()
    rng.shuffle(user_ids)
    devices = []
    start = 0
    for i, size in enumerate(device_sizes):
        users = user_ids[start:start+size]
        start += size
        submat = request_matrix.loc[users].values  # users x items (numpy)
        D_orig = len(users)
        z_orig = int(submat.sum())
        if D_orig <= 1:
            rho = 1.0
        else:
            Xf = submat.astype(float)
            if D_orig > 50:
                idxs = np.random.choice(D_orig, size=50, replace=False)
                Xs = Xf[idxs]
            else:
                Xs = Xf
            sims = cosine_similarity(Xs)
            np.fill_diagonal(sims, 0.0)
            total_pairs = Xs.shape[0] * (Xs.shape[0] - 1)
            rho = float(sims.sum() / total_pairs) if total_pairs > 0 else 0.0
        # hetero params
        f_n = float(np.random.uniform(1.0, 2.0) * 1e9)
        beta_n = float(np.random.uniform(1e-28, 1e-27))
        C_n = int(np.random.randint(5, 10))
        gamma_n = float(np.random.uniform(10, 20))
        t_dn = MODEL_SIZE / (BANDWIDTH_DN * log2(1 + gamma_n))
        wn = 2e6
        t_up = wn / (BANDWIDTH_UP * log2(1 + gamma_n))
        t_comp_coef = (EPOCHS * C_n) / f_n
        e_dn = P_S * t_dn
        e_up = P_N * t_up
        e_comp_coef = beta_n * EPOCHS * C_n * (f_n ** 2)
        devices.append({
            'id': i,
            'users': users,
            'D_orig': D_orig,
            'z_orig': z_orig,
            'rho': rho,
            'f_n': f_n,
            'beta_n': beta_n,
            'C_n': C_n,
            'gamma_n': gamma_n,
            't_dn': t_dn,
            't_up': t_up,
            't_comp_coef': t_comp_coef,
            'e_dn': e_dn,
            'e_up': e_up,
            'e_comp_coef': e_comp_coef,
            'wn': wn
        })
    return devices

# -------------------------
# Helpers
# -------------------------
def device_latency(dev, D):
    return dev['t_dn'] + dev['t_comp_coef'] * D + dev['t_up']

def device_energy(dev, D):
    return dev['e_dn'] + dev['e_comp_coef'] * D + dev['e_up']

def feasible_selected_device(dev, D_val, z_val):
    if D_val > CONSTRAINTS['D_max']: return False
    if z_val < CONSTRAINTS['z_min']: return False
    if device_energy(dev, D_val) > CONSTRAINTS['e_max']: return False
    return True

def feasible_round_latency(selected_ids, D_vals, devices):
    if not selected_ids: return False
    latencies = [device_latency(devices[i], D_vals[i]) for i in selected_ids]
    T = max(latencies) if latencies else 0.0
    return T <= CONSTRAINTS['t_max']

def non_selected_request_ok(z_val, T):
    return (z_val + CONSTRAINTS['alpha_n'] * T) <= CONSTRAINTS['z_max']

# -------------------------
# Softer greedy offloading
# -------------------------
def greedy_offloading_soft(devices, selected_idx, D_vals, z_vals):
    if not selected_idx:
        return np.zeros((len(devices), len(devices)))

    N = len(devices)
    off_matrix = np.zeros((N, N))  # record fractions

    per_unit_z = np.zeros(N)
    for i, d in enumerate(devices):
        per_unit_z[i] = d['z_orig'] / max(1.0, d['D_orig'])

    selset = set(selected_idx)
    non_selected = [i for i in range(N) if i not in selset]
    non_selected_sorted = sorted(non_selected, key=lambda x: devices[x]['D_orig'], reverse=True)

    for src in non_selected_sorted:
        remaining = float(D_vals[src])
        if remaining <= 0 or per_unit_z[src] <= 0:
            continue
        CHUNK = max(1.0, devices[src]['D_orig'] / 20.0)
        while remaining > 1e-9:
            best_gain = 0.0
            best_j = None
            best_chunk = 0.0
            for j in selected_idx:
                if j == src:
                    continue
                cap = CONSTRAINTS['D_max'] - D_vals[j]
                if cap <= 0:
                    continue
                chunk = min(CHUNK, remaining, cap)
                if chunk <= 0:
                    continue
                D_new_j = D_vals[j] + chunk
                z_new_j = z_vals[j] + per_unit_z[src] * chunk
                if not feasible_selected_device(devices[j], D_new_j, z_new_j):
                    continue
                D_temp = D_vals.copy()
                D_temp[j] = D_new_j
                if not feasible_round_latency(selected_idx, D_temp, devices):
                    continue
                gain = devices[j]['rho'] * (per_unit_z[src] * chunk)
                if gain > best_gain:
                    best_gain = gain
                    best_j = j
                    best_chunk = chunk
            if best_j is None:
                if CHUNK > 1.0:
                    CHUNK = max(1.0, CHUNK / 2.0)
                    continue
                else:
                    break
            # round to nearest integer number of users
            best_chunk_int = int(round(best_chunk))
            if best_chunk_int <= 0:
                break  # nothing meaningful to transfer

            # commit transfer with integers
            D_vals[best_j] += best_chunk_int
            z_vals[best_j] += per_unit_z[src] * best_chunk_int
            D_vals[src] = max(0, int(round(D_vals[src] - best_chunk_int)))
            z_vals[src] = max(0.0, z_vals[src] - per_unit_z[src] * best_chunk_int)
            remaining -= best_chunk_int

            # record absolute users offloaded
            off_matrix[src, best_j] += best_chunk_int


    return off_matrix

# -------------------------
# Gym-like environment (with STOP action)
# -------------------------
class FederatedEnv:
    def __init__(self, devices):
        self.devices = devices
        self.N = len(devices)
        self.S_max = CONSTRAINTS['S_max']
        self.S_min = CONSTRAINTS['S_min']
        # action space: 0..N-1 -> pick device, N -> STOP
        self.STOP_ACTION = self.N
        self.reset()

    def reset(self):
        self.D = np.array([d['D_orig'] for d in self.devices], dtype=float)
        self.z = np.array([d['z_orig'] for d in self.devices], dtype=float)
        self.selected = []
        self.done = False
        return self._build_obs()

    def _build_obs(self):
        obs = []
        for i, d in enumerate(self.devices):
            Dn = self.D[i] / max(1.0, CONSTRAINTS['D_max'])
            zn = self.z[i] / max(1.0, CONSTRAINTS['z_max'])
            rho = d['rho']
            en = device_energy(d, self.D[i]) / max(1e-9, CONSTRAINTS['e_max'])
            lat = device_latency(d, self.D[i]) / max(1e-9, CONSTRAINTS['t_max'])
            sel = 1.0 if i in self.selected else 0.0
            obs.extend([Dn, zn, rho, en, lat, sel])
        # append space for STOP_ACTION indicator (we'll not include it per-device, STOP is separate)
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("step after done")
        info = {}
        # If action is STOP
        if action == self.STOP_ACTION:
            # allow STOP only if at least S_min selected
            if len(self.selected) < self.S_min:
                self.done = True
                return self._build_obs(), -10.0, True, info
            # Evaluate final selection + offloading
            self.done = True
            # quick per-selected feasibility check
            if not all(feasible_selected_device(self.devices[i], self.D[i], self.z[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if not feasible_round_latency(self.selected, self.D, self.devices):
                return self._build_obs(), -50.0, True, info
            # Do softer greedy offloading
            D_vals = self.D.copy()
            z_vals = self.z.copy()
            off_matrix = greedy_offloading_soft(self.devices, self.selected, D_vals, z_vals)
            info['off_matrix'] = off_matrix
            T = max(device_latency(self.devices[i], D_vals[i]) for i in self.selected) if self.selected else 0.0
            # non-selected req check
            if not all(non_selected_request_ok(z_vals[i], T) for i in range(self.N) if i not in self.selected):
                return self._build_obs(), -50.0, True, info
            if not all(feasible_selected_device(self.devices[i], D_vals[i], z_vals[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if T > CONSTRAINTS['t_max']:
                return self._build_obs(), -50.0, True, info
            obj = float(sum(z_vals[i] * self.devices[i]['rho'] for i in self.selected))
            reward = obj / 1e6
            info['objective'] = obj
            info['T'] = T
            info['D_vals'] = D_vals
            info['z_vals'] = z_vals
            return self._build_obs(), reward, True, info

        # If action is device pick
        if action < 0 or action >= self.N:
            # invalid action
            self.done = True
            return self._build_obs(), -5.0, True, info

        if action in self.selected:
            # selecting same device twice -> punish
            self.done = True
            return self._build_obs(), -1.0, True, info

        # Add device
        self.selected.append(int(action))

        # If selecting more than allowed, punish and end
        if len(self.selected) > self.S_max:
            self.done = True
            return self._build_obs(), -5.0, True, info

        # Immediate marginal reward: estimate device's raw contribution (rho * z_orig)
        # scaled down so agent can learn numerically
        marginal = self.devices[action]['rho'] * self.devices[action]['z_orig']
        immediate_reward = marginal / 1e6  # small immediate reward

        # If picking this device immediately violates selected-device feasibility, punish and end
        if not feasible_selected_device(self.devices[action], self.D[action], self.z[action]):
            self.done = True
            return self._build_obs(), -20.0, True, info

        # If currently selected devices already violate latency, punish and end
        if not feasible_round_latency(self.selected, self.D, self.devices):
            self.done = True
            return self._build_obs(), -20.0, True, info

        # If haven't hit STOP and haven't filled S_max, episode continues
        # return immediate reward to encourage good picks
        return self._build_obs(), float(immediate_reward), False, info

# -------------------------
# DQN
# -------------------------
if TORCH_OK:
    class ReplayBuffer:
        def __init__(self, capacity=REPLAY_CAPACITY):
            self.buf = deque(maxlen=capacity)
        def push(self, s,a,r,s2,d):
            self.buf.append((s,a,r,s2,d))
        def sample(self, batch_size):
            batch = random.sample(self.buf, batch_size)
            s,a,r,s2,d = map(np.array, zip(*batch))
            return s,a,r,s2,d
        def __len__(self):
            return len(self.buf)

    class QNet(nn.Module):
        def __init__(self, in_dim, out_dim, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim)
            )
        def forward(self, x): return self.net(x)

    class DQNAgent:
        def __init__(self, obs_dim, n_actions, lr=LR):
            self.q = QNet(obs_dim, n_actions)
            self.target = QNet(obs_dim, n_actions)
            self.target.load_state_dict(self.q.state_dict())
            self.opt = optim.Adam(self.q.parameters(), lr=lr)
            self.replay = ReplayBuffer()
            self.eps = EPS_START
            self.n_actions = n_actions
            self.step_count = 0

        def select(self, state):
            if random.random() < self.eps:
                return random.randrange(self.n_actions)
            s = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                qvals = self.q(s).cpu().numpy()[0]
            return int(np.argmax(qvals))

        def store(self, s,a,r,s2,d):
            self.replay.push(s,a,r,s2,d)

        def update(self, batch_size=BATCH_SIZE):
            if len(self.replay) < batch_size:
                return
            s,a,r,s2,d = self.replay.sample(batch_size)
            s = torch.FloatTensor(s)
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r)
            s2 = torch.FloatTensor(s2)
            d = torch.FloatTensor(d)
            qvals = self.q(s).gather(1, a).squeeze()
            with torch.no_grad():
                qnext = self.target(s2).max(1)[0]
                target = r + (1-d) * GAMMA * qnext
            loss = nn.MSELoss()(qvals, target)
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
            self.step_count += 1
            self.eps = max(EPS_MIN, self.eps * EPS_DECAY)
            if self.step_count % 500 == 0:
                self.target.load_state_dict(self.q.state_dict())

# -------------------------
# Training loop
# -------------------------
def train_dqn_on_movielens(devices, episodes=EPISODES):
    env = FederatedEnv(devices)
    obs0 = env.reset()
    obs_dim = obs0.shape[0]
    n_actions = len(devices) + 1  # extra STOP action
    if not TORCH_OK:
        raise RuntimeError("PyTorch not installed. Install torch to train DQN.")
    agent = DQNAgent(obs_dim, n_actions, lr=LR)
    best = {'obj': -1.0, 'info': None, 'selected': None}
    rewards = []
    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action = agent.select(state)
            next_s, r, done, info = env.step(action)
            agent.store(state, action, r, next_s, float(done))
            agent.update(BATCH_SIZE)
            state = next_s
            ep_reward += r
            # if done, break; else continue picking/stop
        rewards.append(ep_reward)
        if info and 'objective' in info:
            obj = info['objective']
            if obj > best['obj']:
                best['obj'] = obj
                best['info'] = info
                best['selected'] = env.selected.copy()
        if (ep+1) % 50 == 0:
            print(f"Episode {ep+1}/{episodes} — ep_reward={ep_reward:.3f} eps={agent.eps:.3f} best_obj={best['obj']:.1f}")
    return agent, best, rewards

# -------------------------
# Random device sizes helper
# -------------------------
def get_random_device_sizes(num_users, num_devices=10, seed=None):
    rng = np.random.default_rng(seed)
    probs = np.ones(num_devices) / num_devices
    sizes = rng.multinomial(num_users, probs)
    return sizes.tolist()

# -------------------------
# Detailed solution print
# -------------------------
def print_solution_details(best, devices):
    if best['info'] is None:
        print("No feasible solution found.")
        return
    selected = best['selected']
    D_vals = best['info']['D_vals']
    z_vals = best['info']['z_vals']
    T = best['info']['T']
    print("\n=== DETAILED SOLUTION STATS ===")
    print(f"Selected devices ({len(selected)}): {selected}")
    print(f"Objective: {best['info']['objective']:.2f}")
    print(f"Round latency T: {T:.6f} s")
    total_users = 0.0
    total_requests = 0.0
    total_energy = 0.0
    for i, dev in enumerate(devices):
        sel = "✅" if i in selected else "❌"
        D = D_vals[i]
        z = z_vals[i]
        rho = dev['rho']
        lat = device_latency(dev, D)
        ene = device_energy(dev, D)
        total_users += D
        total_requests += z
        total_energy += ene
        print(f" Device {i:2d} {sel} | Users_orig={dev['D_orig']:4d} | D_final={D:7.1f} | z_final={z:8.1f} | rho={rho:.4f} | Lat={lat:.6f}s | E={ene:.4f}J")
    print("\nTotals: Users(sum)={:.1f} | Requests(sum)={:.1f} | Energy(sum)={:.4f}J".format(total_users, total_requests, total_energy))
    # check constraints
    lat_ok = T <= CONSTRAINTS['t_max']
    s_ok = (CONSTRAINTS['S_min'] <= len(selected) <= CONSTRAINTS['S_max'])
    print(f"Constraint checks: Latency OK={lat_ok} | Selection count OK={s_ok}")
    if 'off_matrix' in best['info']:
        print("\n=== OFFLOADING MATRIX (users) ===")
        off = best['info']['off_matrix']
        df_off = pd.DataFrame(off, index=[f"Dev{i}" for i in range(len(devices))], columns=[f"Dev{j}" for j in range(len(devices))])
        print(df_off.round(1))


# -------------------------
# Main
# -------------------------
def main():
    print("Loading MovieLens top-1000 movies and creating devices...")
    request_matrix = load_movielens_binary_matrix(MOVIELENS_PATH, top_k_movies=1000)
    num_users = request_matrix.shape[0]
    num_devices = 10
    device_sizes = get_random_device_sizes(num_users, num_devices)
    # print("Random device sizes:", device_sizes, " (sum =", sum(device_sizes), ")")
    devices = create_devices_from_matrix(request_matrix, device_sizes)
    # print("Devices prepared. D_orig list:", [d['D_orig'] for d in devices])
    if not TORCH_OK:
        print("PyTorch not available. Install torch and re-run to train DQN.")
        return
    agent, best, rewards = train_dqn_on_movielens(devices, episodes=EPISODES)
    print("\n=== BEST SOLUTION FOUND ===")
    print_solution_details(best, devices)

if __name__ == "__main__":
    main()


    """
    FL device selection (MovieLens) + improved DQN agent for device selection.
    - STOP action added (so agent chooses number of devices).
    - Softer greedy offloading (partial chunks).
    - Immediate marginal reward on selection.
    - Prints detailed solution stats at the end.

    MODIFIED: removed downsampling in cosine-sim calculation (use all users per device)
    and replaced np.random.* calls inside create_devices_from_matrix with local RNG for
    reproducible behavior controlled by the `seed` parameter.
    """

    import math
    import random
    import time
    from collections import deque

    import numpy as np
    import pandas as pd
    from sklearn.metrics.pairwise import cosine_similarity

    # PyTorch for DQN
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
        TORCH_OK = True
    except Exception:
        TORCH_OK = False

    # -------------------------
    # Configuration / constants
    # -------------------------
    MOVIELENS_PATH = r'C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m'  # change if needed

    MODEL_SIZE = 5e6      # bits (example)
    BANDWIDTH_DN = 50e6   # bps
    BANDWIDTH_UP = 10e6   # bps
    P_S = 0.5
    P_N = 0.3
    EPOCHS = 1
    # HYPERPARAMETERS DEFINED

    CONSTRAINTS = {
        'z_min': 50,
        'z_max': 5000,
        'D_max': 2000,
        't_max': 0.1,
        'e_max': 0.5,
        'S_min': 2,
        'S_max': 9,
        'alpha_n': 0.1
    }
    # CONSTRAINTS DEFINED

    # DQN hyperparameters
    EPISODES = 400
    BATCH_SIZE = 64
    LR = 1e-3
    GAMMA = 0.99
    EPS_START = 1.0
    EPS_MIN = 0.05
    EPS_DECAY = 0.995
    REPLAY_CAPACITY = 20000
    SEED = 1

    np.random.seed(SEED)
    random.seed(SEED)
    if TORCH_OK:
        torch.manual_seed(SEED)

    # -------------------------
    # Utilities
    # -------------------------
    def log2(x):
        return math.log(x, 2.0)

    # -------------------------
    # MovieLens loader + preprocessing
    # -------------------------
    def load_movielens_binary_matrix(path_dir, top_k_movies=1000):
        ratings_path = path_dir.rstrip('/') + '/ratings.dat'
        ratings_col = ['UserID', 'MovieID', 'Rating', 'Timestamp']
        print("Loading ratings from:", ratings_path)
        ratings = pd.read_csv(ratings_path, sep='::', names=ratings_col, engine='python')
        top_movies = ratings['MovieID'].value_counts().nlargest(top_k_movies).index
        ratings = ratings[ratings['MovieID'].isin(top_movies)]
        request_matrix = ratings.pivot_table(index='UserID', columns='MovieID', values='Rating', fill_value=0)
        request_matrix = (request_matrix > 0).astype(int)
        print("Request matrix shape (users x movies):", request_matrix.shape)
        return request_matrix

    # -------------------------
    # Build devices from request matrix
    # -------------------------
    def create_devices_from_matrix(request_matrix, device_sizes, seed=0):
        """
        Create a list of device dictionaries from the request matrix.

        Changes made here:
        - Removed the downsampling that limited similarity computation to 50 users.
        Now we compute cosine similarity using *all* users assigned to a device.
        - Use a local RNG (np.random.default_rng) for all random draws inside this
        function so behavior is reproducible with the provided `seed`.
        """
        rng = np.random.default_rng(seed)
        user_ids = request_matrix.index.tolist()
        rng.shuffle(user_ids)
        devices = []
        start = 0
        for i, size in enumerate(device_sizes):
            users = user_ids[start:start+size]
            start += size
            submat = request_matrix.loc[users].values  # users x items (numpy)
            D_orig = len(users)
            z_orig = int(submat.sum())
            if D_orig <= 1:
                rho = 1.0
            else:
                # take all users (no downsampling)
                Xf = submat.astype(float)
                Xs = Xf  # use full device user set for similarity
                # compute cosine similarity among all users in device
                sims = cosine_similarity(Xs)
                np.fill_diagonal(sims, 0.0)
                total_pairs = Xs.shape[0] * (Xs.shape[0] - 1)
                rho = float(sims.sum() / total_pairs) if total_pairs > 0 else 0.0
            # hetero params (drawn from local RNG for reproducibility)
            f_n = float(rng.uniform(1.0, 2.0) * 1e9)
            beta_n = float(rng.uniform(1e-28, 1e-27))
            C_n = int(rng.integers(5, 10))
            gamma_n = float(rng.uniform(10, 20))
            t_dn = MODEL_SIZE / (BANDWIDTH_DN * log2(1 + gamma_n))
            wn = 2e6
            t_up = wn / (BANDWIDTH_UP * log2(1 + gamma_n))
            t_comp_coef = (EPOCHS * C_n) / f_n
            e_dn = P_S * t_dn
            e_up = P_N * t_up
            e_comp_coef = beta_n * EPOCHS * C_n * (f_n ** 2)
            devices.append({
                'id': i,
                'users': users,
                'D_orig': D_orig,
                'z_orig': z_orig,
                'rho': rho,
                'f_n': f_n,
                'beta_n': beta_n,
                'C_n': C_n,
                'gamma_n': gamma_n,
                't_dn': t_dn,
                't_up': t_up,
                't_comp_coef': t_comp_coef,
                'e_dn': e_dn,
                'e_up': e_up,
                'e_comp_coef': e_comp_coef,
                'wn': wn
            })
        return devices

    # -------------------------
    # Helpers
    # -------------------------
    def device_latency(dev, D):
        return dev['t_dn'] + dev['t_comp_coef'] * D + dev['t_up']

    def device_energy(dev, D):
        return dev['e_dn'] + dev['e_comp_coef'] * D + dev['e_up']

    def feasible_selected_device(dev, D_val, z_val):
        if D_val > CONSTRAINTS['D_max']: return False
        if z_val < CONSTRAINTS['z_min']: return False
        if device_energy(dev, D_val) > CONSTRAINTS['e_max']: return False
        return True

    def feasible_round_latency(selected_ids, D_vals, devices):
        if not selected_ids: return False
        latencies = [device_latency(devices[i], D_vals[i]) for i in selected_ids]
        T = max(latencies) if latencies else 0.0
        return T <= CONSTRAINTS['t_max']

    def non_selected_request_ok(z_val, T):
        return (z_val + CONSTRAINTS['alpha_n'] * T) <= CONSTRAINTS['z_max']

    # -------------------------
    # Softer greedy offloading
    # -------------------------
    def greedy_offloading_soft(devices, selected_idx, D_vals, z_vals):
        if not selected_idx:
            return np.zeros((len(devices), len(devices)))

        N = len(devices)
        off_matrix = np.zeros((N, N))  # record fractions

        per_unit_z = np.zeros(N)
        for i, d in enumerate(devices):
            per_unit_z[i] = d['z_orig'] / max(1.0, d['D_orig'])

        selset = set(selected_idx)
        non_selected = [i for i in range(N) if i not in selset]
        non_selected_sorted = sorted(non_selected, key=lambda x: devices[x]['D_orig'], reverse=True)

        for src in non_selected_sorted:
            remaining = float(D_vals[src])
            if remaining <= 0 or per_unit_z[src] <= 0:
                continue
            CHUNK = max(1.0, devices[src]['D_orig'] / 20.0)
            while remaining > 1e-9:
                best_gain = 0.0
                best_j = None
                best_chunk = 0.0
                for j in selected_idx:
                    if j == src:
                        continue
                    cap = CONSTRAINTS['D_max'] - D_vals[j]
                    if cap <= 0:
                        continue
                    chunk = min(CHUNK, remaining, cap)
                    if chunk <= 0:
                        continue
                    D_new_j = D_vals[j] + chunk
                    z_new_j = z_vals[j] + per_unit_z[src] * chunk
                    if not feasible_selected_device(devices[j], D_new_j, z_new_j):
                        continue
                    D_temp = D_vals.copy()
                    D_temp[j] = D_new_j
                    if not feasible_round_latency(selected_idx, D_temp, devices):
                        continue
                    gain = devices[j]['rho'] * (per_unit_z[src] * chunk)
                    if gain > best_gain:
                        best_gain = gain
                        best_j = j
                        best_chunk = chunk
                if best_j is None:
                    if CHUNK > 1.0:
                        CHUNK = max(1.0, CHUNK / 2.0)
                        continue
                    else:
                        break
                # round to nearest integer number of users
                best_chunk_int = int(round(best_chunk))
                if best_chunk_int <= 0:
                    break  # nothing meaningful to transfer

                # commit transfer with integers
                D_vals[best_j] += best_chunk_int
                z_vals[best_j] += per_unit_z[src] * best_chunk_int
                D_vals[src] = max(0, int(round(D_vals[src] - best_chunk_int)))
                z_vals[src] = max(0.0, z_vals[src] - per_unit_z[src] * best_chunk_int)
                remaining -= best_chunk_int

                # record absolute users offloaded
                off_matrix[src, best_j] += best_chunk_int


        return off_matrix

    # -------------------------
    # Gym-like environment (with STOP action)
    # -------------------------
    class FederatedEnv:
        def __init__(self, devices):
            self.devices = devices
            self.N = len(devices)
            self.S_max = CONSTRAINTS['S_max']
            self.S_min = CONSTRAINTS['S_min']
            # action space: 0..N-1 -> pick device, N -> STOP
            self.STOP_ACTION = self.N
            self.reset()

        def reset(self):
            self.D = np.array([d['D_orig'] for d in self.devices], dtype=float)
            self.z = np.array([d['z_orig'] for d in self.devices], dtype=float)
            self.selected = []
            self.done = False
            return self._build_obs()

        def _build_obs(self):
            obs = []
            for i, d in enumerate(self.devices):
                Dn = self.D[i] / max(1.0, CONSTRAINTS['D_max'])
                zn = self.z[i] / max(1.0, CONSTRAINTS['z_max'])
                rho = d['rho']
                en = device_energy(d, self.D[i]) / max(1e-9, CONSTRAINTS['e_max'])
                lat = device_latency(d, self.D[i]) / max(1e-9, CONSTRAINTS['t_max'])
                sel = 1.0 if i in self.selected else 0.0
                obs.extend([Dn, zn, rho, en, lat, sel])
            # append space for STOP_ACTION indicator (we'll not include it per-device, STOP is separate)
            return np.array(obs, dtype=np.float32)

        def step(self, action):
            if self.done:
                raise RuntimeError("step after done")
            info = {}
            # If action is STOP
            if action == self.STOP_ACTION:
                # allow STOP only if at least S_min selected
                if len(self.selected) < self.S_min:
                    self.done = True
                    return self._build_obs(), -10.0, True, info
                # Evaluate final selection + offloading
                self.done = True
                # quick per-selected feasibility check
                if not all(feasible_selected_device(self.devices[i], self.D[i], self.z[i]) for i in self.selected):
                    return self._build_obs(), -50.0, True, info
                if not feasible_round_latency(self.selected, self.D, self.devices):
                    return self._build_obs(), -50.0, True, info
                # Do softer greedy offloading
                D_vals = self.D.copy()
                z_vals = self.z.copy()
                off_matrix = greedy_offloading_soft(self.devices, self.selected, D_vals, z_vals)
                info['off_matrix'] = off_matrix
                T = max(device_latency(self.devices[i], D_vals[i]) for i in self.selected) if self.selected else 0.0
                # non-selected req check
                if not all(non_selected_request_ok(z_vals[i], T) for i in range(self.N) if i not in self.selected):
                    return self._build_obs(), -50.0, True, info
                if not all(feasible_selected_device(self.devices[i], D_vals[i], z_vals[i]) for i in self.selected):
                    return self._build_obs(), -50.0, True, info
                if T > CONSTRAINTS['t_max']:
                    return self._build_obs(), -50.0, True, info
                obj = float(sum(z_vals[i] * self.devices[i]['rho'] for i in self.selected))
                reward = obj / 1e6
                info['objective'] = obj
                info['T'] = T
                info['D_vals'] = D_vals
                info['z_vals'] = z_vals
                return self._build_obs(), reward, True, info

            # If action is device pick
            if action < 0 or action >= self.N:
                # invalid action
                self.done = True
                return self._build_obs(), -5.0, True, info

            if action in self.selected:
                # selecting same device twice -> punish
                self.done = True
                return self._build_obs(), -1.0, True, info

            # Add device
            self.selected.append(int(action))

            # If selecting more than allowed, punish and end
            if len(self.selected) > self.S_max:
                self.done = True
                return self._build_obs(), -5.0, True, info

            # Immediate marginal reward: estimate device's raw contribution (rho * z_orig)
            # scaled down so agent can learn numerically
            marginal = self.devices[action]['rho'] * self.devices[action]['z_orig']
            immediate_reward = marginal / 1e6  # small immediate reward

            # If picking this device immediately violates selected-device feasibility, punish and end
            if not feasible_selected_device(self.devices[action], self.D[action], self.z[action]):
                self.done = True
                return self._build_obs(), -20.0, True, info

            # If currently selected devices already violate latency, punish and end
            if not feasible_round_latency(self.selected, self.D, self.devices):
                self.done = True
                return self._build_obs(), -20.0, True, info

            # If haven't hit STOP and haven't filled S_max, episode continues
            # return immediate reward to encourage good picks
            return self._build_obs(), float(immediate_reward), False, info

    # -------------------------
    # DQN
    # -------------------------
    if TORCH_OK:
        class ReplayBuffer:
            def __init__(self, capacity=REPLAY_CAPACITY):
                self.buf = deque(maxlen=capacity)
            def push(self, s,a,r,s2,d):
                self.buf.append((s,a,r,s2,d))
            def sample(self, batch_size):
                batch = random.sample(self.buf, batch_size)
                s,a,r,s2,d = map(np.array, zip(*batch))
                return s,a,r,s2,d
            def __len__(self):
                return len(self.buf)

        class QNet(nn.Module):
            def __init__(self, in_dim, out_dim, hidden=256):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, hidden), nn.ReLU(),
                    nn.Linear(hidden, hidden), nn.ReLU(),
                    nn.Linear(hidden, out_dim)
                )
            def forward(self, x): return self.net(x)

        class DQNAgent:
            def __init__(self, obs_dim, n_actions, lr=LR):
                self.q = QNet(obs_dim, n_actions)
                self.target = QNet(obs_dim, n_actions)
                self.target.load_state_dict(self.q.state_dict())
                self.opt = optim.Adam(self.q.parameters(), lr=lr)
                self.replay = ReplayBuffer()
                self.eps = EPS_START
                self.n_actions = n_actions
                self.step_count = 0

            def select(self, state):
                if random.random() < self.eps:
                    return random.randrange(self.n_actions)
                s = torch.FloatTensor(state).unsqueeze(0)
                with torch.no_grad():
                    qvals = self.q(s).cpu().numpy()[0]
                return int(np.argmax(qvals))

            def store(self, s,a,r,s2,d):
                self.replay.push(s,a,r,s2,d)

            def update(self, batch_size=BATCH_SIZE):
                if len(self.replay) < batch_size:
                    return
                s,a,r,s2,d = self.replay.sample(batch_size)
                s = torch.FloatTensor(s)
                a = torch.LongTensor(a).unsqueeze(1)
                r = torch.FloatTensor(r)
                s2 = torch.FloatTensor(s2)
                d = torch.FloatTensor(d)
                qvals = self.q(s).gather(1, a).squeeze()
                with torch.no_grad():
                    qnext = self.target(s2).max(1)[0]
                    target = r + (1-d) * GAMMA * qnext
                loss = nn.MSELoss()(qvals, target)
                self.opt.zero_grad()
                loss.backward()
                self.opt.step()
                self.step_count += 1
                self.eps = max(EPS_MIN, self.eps * EPS_DECAY)
                if self.step_count % 500 == 0:
                    self.target.load_state_dict(self.q.state_dict())

    # -------------------------
    # Training loop
    # -------------------------
    def train_dqn_on_movielens(devices, episodes=EPISODES):
        env = FederatedEnv(devices)
        obs0 = env.reset()
        obs_dim = obs0.shape[0]
        n_actions = len(devices) + 1  # extra STOP action
        if not TORCH_OK:
            raise RuntimeError("PyTorch not installed. Install torch to train DQN.")
        agent = DQNAgent(obs_dim, n_actions, lr=LR)
        best = {'obj': -1.0, 'info': None, 'selected': None}
        rewards = []
        for ep in range(episodes):
            state = env.reset()
            done = False
            ep_reward = 0.0
            while not done:
                action = agent.select(state)
                next_s, r, done, info = env.step(action)
                agent.store(state, action, r, next_s, float(done))
                agent.update(BATCH_SIZE)
                state = next_s
                ep_reward += r
                # if done, break; else continue picking/stop
            rewards.append(ep_reward)
            if info and 'objective' in info:
                obj = info['objective']
                if obj > best['obj']:
                    best['obj'] = obj
                    best['info'] = info
                    best['selected'] = env.selected.copy()
            if (ep+1) % 50 == 0:
                print(f"Episode {ep+1}/{episodes} — ep_reward={ep_reward:.3f} eps={agent.eps:.3f} best_obj={best['obj']:.1f}")
        return agent, best, rewards

    # -------------------------
    # Random device sizes helper
    # -------------------------
    def get_random_device_sizes(num_users, num_devices=10, seed=None):
        rng = np.random.default_rng(seed)
        probs = np.ones(num_devices) / num_devices
        sizes = rng.multinomial(num_users, probs)
        return sizes.tolist()

    # -------------------------
    # Detailed solution print
    # -------------------------
    def print_solution_details(best, devices):
        if best['info'] is None:
            print("No feasible solution found.")
            return
        selected = best['selected']
        D_vals = best['info']['D_vals']
        z_vals = best['info']['z_vals']
        T = best['info']['T']
        print("\n=== DETAILED SOLUTION STATS ===")
        print(f"Selected devices ({len(selected)}): {selected}")
        print(f"Objective: {best['info']['objective']:.2f}")
        print(f"Round latency T: {T:.6f} s")
        total_users = 0.0
        total_requests = 0.0
        total_energy = 0.0
        for i, dev in enumerate(devices):
            sel = "✅" if i in selected else "❌"
            D = D_vals[i]
            z = z_vals[i]
            rho = dev['rho']
            lat = device_latency(dev, D)
            ene = device_energy(dev, D)
            total_users += D
            total_requests += z
            total_energy += ene
            print(f" Device {i:2d} {sel} | Users_orig={dev['D_orig']:4d} | D_final={D:7.1f} | z_final={z:8.1f} | rho={rho:.4f} | Lat={lat:.6f}s | E={ene:.4f}J")
        print("\nTotals: Users(sum)={:.1f} | Requests(sum)={:.1f} | Energy(sum)={:.4f}J".format(total_users, total_requests, total_energy))
        # check constraints
        lat_ok = T <= CONSTRAINTS['t_max']
        s_ok = (CONSTRAINTS['S_min'] <= len(selected) <= CONSTRAINTS['S_max'])
        print(f"Constraint checks: Latency OK={lat_ok} | Selection count OK={s_ok}")
        if 'off_matrix' in best['info']:
            print("\n=== OFFLOADING MATRIX (users) ===")
            off = best['info']['off_matrix']
            df_off = pd.DataFrame(off, index=[f"Dev{i}" for i in range(len(devices))], columns=[f"Dev{j}" for j in range(len(devices))])
            print(df_off.round(1))


    # -------------------------
    # Main
    # -------------------------
    def main():
        print("Loading MovieLens top-1000 movies and creating devices...")
        request_matrix = load_movielens_binary_matrix(MOVIELENS_PATH, top_k_movies=1000)
        num_users = request_matrix.shape[0]
        num_devices = 10
        device_sizes = get_random_device_sizes(num_users, num_devices)
        # print("Random device sizes:", device_sizes, " (sum =", sum(device_sizes), ")")
        devices = create_devices_from_matrix(request_matrix, device_sizes)
        # print("Devices prepared. D_orig list:", [d['D_orig'] for d in devices])
        if not TORCH_OK:
            print("PyTorch not available. Install torch and re-run to train DQN.")
            return
        agent, best, rewards = train_dqn_on_movielens(devices, episodes=EPISODES)
        print("\n=== BEST SOLUTION FOUND ===")
        print_solution_details(best, devices)

    if __name__ == "__main__":
        main()


"""
FL device selection (MovieLens) + improved DQN agent for device selection.
- STOP action added (so agent chooses number of devices).
- Softer greedy offloading (partial chunks).
- Immediate marginal reward on selection.
- Prints detailed solution stats at the end.

MODIFIED: removed downsampling in cosine-sim calculation (use all users per device)
and replaced np.random.* calls inside create_devices_from_matrix with local RNG for
reproducible behavior controlled by the `seed` parameter.

MODIFIED AGAIN: Replaced expensive on-the-fly rho recalculation with a faster
heuristic based on a weighted average of the offloaded content's source rho.
"""

import math
import random
import time
from collections import deque

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# PyTorch for DQN
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# -------------------------
# Configuration / constants
# -------------------------
MOVIELENS_PATH = r'C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m'  # change if needed

MODEL_SIZE = 5e6      # bits (example)
BANDWIDTH_DN = 50e6   # bps
BANDWIDTH_UP = 10e6   # bps
P_S = 0.5
P_N = 0.3
EPOCHS = 1
# HYPERPARAMETERS DEFINED

CONSTRAINTS = {
    'z_min': 50,
    'z_max': 5000,
    'D_max': 2000,
    't_max': 0.1,
    'e_max': 0.5,
    'S_min': 2,
    'S_max': 9,
    'alpha_n': 0.1
}
# CONSTRAINTS DEFINED

# DQN hyperparameters
EPISODES = 400
BATCH_SIZE = 64
LR = 1e-3
GAMMA = 0.99
EPS_START = 1.0
EPS_MIN = 0.05
EPS_DECAY = 0.995
REPLAY_CAPACITY = 20000
SEED = 1

np.random.seed(SEED)
random.seed(SEED)
if TORCH_OK:
    torch.manual_seed(SEED)

# -------------------------
# Utilities
# -------------------------
def log2(x):
    return math.log(x, 2.0)

# -------------------------
# MovieLens loader + preprocessing
# -------------------------
def load_movielens_binary_matrix(path_dir, top_k_movies=1000):
    ratings_path = path_dir.rstrip('/') + '/ratings.dat'
    ratings_col = ['UserID', 'MovieID', 'Rating', 'Timestamp']
    print("Loading ratings from:", ratings_path)
    ratings = pd.read_csv(ratings_path, sep='::', names=ratings_col, engine='python')
    top_movies = ratings['MovieID'].value_counts().nlargest(top_k_movies).index
    ratings = ratings[ratings['MovieID'].isin(top_movies)]
    request_matrix = ratings.pivot_table(index='UserID', columns='MovieID', values='Rating', fill_value=0)
    request_matrix = (request_matrix > 0).astype(int)
    print("Request matrix shape (users x movies):", request_matrix.shape)
    return request_matrix

# -------------------------
# Build devices from request matrix
# -------------------------
def create_devices_from_matrix(request_matrix, device_sizes, seed=0):
    """
    Create a list of device dictionaries from the request matrix.

    Changes made here:
    - Removed the downsampling that limited similarity computation to 50 users.
      Now we compute cosine similarity using *all* users assigned to a device.
    - Use a local RNG (np.random.default_rng) for all random draws inside this
      function so behavior is reproducible with the provided `seed`.
    """
    rng = np.random.default_rng(seed)
    user_ids = request_matrix.index.tolist()
    rng.shuffle(user_ids)
    devices = []
    start = 0
    for i, size in enumerate(device_sizes):
        users = user_ids[start:start+size]
        start += size
        submat = request_matrix.loc[users].values  # users x items (numpy)
        D_orig = len(users)
        z_orig = int(submat.sum())
        if D_orig <= 1:
            rho = 1.0
        else:
            # take all users (no downsampling)
            Xf = submat.astype(float)
            Xs = Xf  # use full device user set for similarity
            # compute cosine similarity among all users in device
            sims = cosine_similarity(Xs)
            np.fill_diagonal(sims, 0.0)
            total_pairs = Xs.shape[0] * (Xs.shape[0] - 1)
            rho = float(sims.sum() / total_pairs) if total_pairs > 0 else 0.0
        # hetero params (drawn from local RNG for reproducibility)
        f_n = float(rng.uniform(1.0, 2.0) * 1e9)
        beta_n = float(rng.uniform(1e-28, 1e-27))
        C_n = int(rng.integers(5, 10))
        gamma_n = float(rng.uniform(10, 20))
        t_dn = MODEL_SIZE / (BANDWIDTH_DN * log2(1 + gamma_n))
        wn = 2e6
        t_up = wn / (BANDWIDTH_UP * log2(1 + gamma_n))
        t_comp_coef = (EPOCHS * C_n) / f_n
        e_dn = P_S * t_dn
        e_up = P_N * t_up
        e_comp_coef = beta_n * EPOCHS * C_n * (f_n ** 2)
        devices.append({
            'id': i,
            'users': users,
            'D_orig': D_orig,
            'z_orig': z_orig,
            'rho': rho,
            'f_n': f_n,
            'beta_n': beta_n,
            'C_n': C_n,
            'gamma_n': gamma_n,
            't_dn': t_dn,
            't_up': t_up,
            't_comp_coef': t_comp_coef,
            'e_dn': e_dn,
            'e_up': e_up,
            'e_comp_coef': e_comp_coef,
            'wn': wn
        })
    return devices

# -------------------------
# Helpers
# -------------------------
def device_latency(dev, D):
    return dev['t_dn'] + dev['t_comp_coef'] * D + dev['t_up']

def device_energy(dev, D):
    return dev['e_dn'] + dev['e_comp_coef'] * D + dev['e_up']

def feasible_selected_device(dev, D_val, z_val):
    if D_val > CONSTRAINTS['D_max']: return False
    if z_val < CONSTRAINTS['z_min']: return False
    if device_energy(dev, D_val) > CONSTRAINTS['e_max']: return False
    return True

def feasible_round_latency(selected_ids, D_vals, devices):
    if not selected_ids: return False
    latencies = [device_latency(devices[i], D_vals[i]) for i in selected_ids]
    T = max(latencies) if latencies else 0.0
    return T <= CONSTRAINTS['t_max']

def non_selected_request_ok(z_val, T):
    return (z_val + CONSTRAINTS['alpha_n'] * T) <= CONSTRAINTS['z_max']

# -------------------------
# Softer greedy offloading
# -------------------------
def greedy_offloading_soft(devices, selected_idx, D_vals, z_vals):
    if not selected_idx:
        return np.zeros((len(devices), len(devices))), {}

    N = len(devices)
    off_matrix = np.zeros((N, N))

    # IMPORTANT: Track the current rho values for each device
    current_rhos = {i: d['rho'] for i, d in enumerate(devices)}

    per_unit_z = np.zeros(N)
    for i, d in enumerate(devices):
        per_unit_z[i] = d['z_orig'] / max(1.0, d['D_orig'])

    selset = set(selected_idx)
    non_selected = [i for i in range(N) if i not in selset]
    non_selected_sorted = sorted(non_selected, key=lambda x: devices[x]['D_orig'], reverse=True)

    for src in non_selected_sorted:
        remaining = float(D_vals[src])
        if remaining <= 0 or per_unit_z[src] <= 0:
            continue
        
        CHUNK = max(1.0, devices[src]['D_orig'] / 20.0)
        while remaining > 1e-9:
            best_gain = 0.0
            best_j = None
            best_chunk = 0.0
            
            for j in selected_idx:
                if j == src: continue
                
                cap = CONSTRAINTS['D_max'] - D_vals[j]
                if cap <= 0: continue
                
                chunk = min(CHUNK, remaining, cap)
                if chunk <= 0: continue
                
                D_new_j = D_vals[j] + chunk
                z_new_j = z_vals[j] + per_unit_z[src] * chunk

                if not feasible_selected_device(devices[j], D_new_j, z_new_j): continue
                
                D_temp = D_vals.copy()
                D_temp[j] = D_new_j
                if not feasible_round_latency(selected_idx, D_temp, devices): continue

                # Gain is calculated with the DESTINATION's current rho
                gain = current_rhos[j] * (per_unit_z[src] * chunk)
                
                if gain > best_gain:
                    best_gain = gain
                    best_j = j
                    best_chunk = chunk
            
            if best_j is None:
                if CHUNK > 1.0:
                    CHUNK = max(1.0, CHUNK / 2.0)
                    continue
                else:
                    break
            
            best_chunk_int = int(round(best_chunk))
            if best_chunk_int <= 0: break

            # --- NEW: RHO UPDATE LOGIC ---
            z_moved = per_unit_z[src] * best_chunk_int
            
            # Get old state of destination device
            old_rho_dest = current_rhos[best_j]
            old_z_dest = z_vals[best_j]
            
            # Get rho of the source device
            rho_src = current_rhos[src]
            
            # Apply the weighted average formula
            numerator = (old_z_dest * old_rho_dest) + (z_moved * rho_src)
            denominator = old_z_dest + z_moved
            
            if denominator > 0:
                new_rho_dest = numerator / denominator
                current_rhos[best_j] = new_rho_dest
            # --- END OF NEW LOGIC ---

            # Commit the transfer
            D_vals[best_j] += best_chunk_int
            z_vals[best_j] += z_moved
            D_vals[src] = max(0, int(round(D_vals[src] - best_chunk_int)))
            z_vals[src] = max(0.0, z_vals[src] - per_unit_z[src] * best_chunk_int)
            remaining -= best_chunk_int
            
            off_matrix[src, best_j] += best_chunk_int

    return off_matrix, current_rhos

# -------------------------
# Gym-like environment (with STOP action)
# -------------------------
class FederatedEnv:
    def __init__(self, devices):
        self.devices = devices
        self.N = len(devices)
        self.S_max = CONSTRAINTS['S_max']
        self.S_min = CONSTRAINTS['S_min']
        # action space: 0..N-1 -> pick device, N -> STOP
        self.STOP_ACTION = self.N
        self.reset()

    def reset(self):
        self.D = np.array([d['D_orig'] for d in self.devices], dtype=float)
        self.z = np.array([d['z_orig'] for d in self.devices], dtype=float)
        self.selected = []
        self.done = False
        return self._build_obs()

    def _build_obs(self):
        obs = []
        for i, d in enumerate(self.devices):
            Dn = self.D[i] / max(1.0, CONSTRAINTS['D_max'])
            zn = self.z[i] / max(1.0, CONSTRAINTS['z_max'])
            rho = d['rho']
            en = device_energy(d, self.D[i]) / max(1e-9, CONSTRAINTS['e_max'])
            lat = device_latency(d, self.D[i]) / max(1e-9, CONSTRAINTS['t_max'])
            sel = 1.0 if i in self.selected else 0.0
            obs.extend([Dn, zn, rho, en, lat, sel])
        # append space for STOP_ACTION indicator (we'll not include it per-device, STOP is separate)
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("step after done")
        info = {}
        # If action is STOP
        if action == self.STOP_ACTION:
            # allow STOP only if at least S_min selected
            if len(self.selected) < self.S_min:
                self.done = True
                return self._build_obs(), -10.0, True, info
            # Evaluate final selection + offloading
            self.done = True
            # quick per-selected feasibility check
            if not all(feasible_selected_device(self.devices[i], self.D[i], self.z[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if not feasible_round_latency(self.selected, self.D, self.devices):
                return self._build_obs(), -50.0, True, info
            # Do softer greedy offloading
            D_vals = self.D.copy()
            z_vals = self.z.copy()
            off_matrix, final_rhos = greedy_offloading_soft(self.devices, self.selected, D_vals, z_vals)
            info['off_matrix'] = off_matrix
            T = max(device_latency(self.devices[i], D_vals[i]) for i in self.selected) if self.selected else 0.0
            # non-selected req check
            if not all(non_selected_request_ok(z_vals[i], T) for i in range(self.N) if i not in self.selected):
                return self._build_obs(), -50.0, True, info
            if not all(feasible_selected_device(self.devices[i], D_vals[i], z_vals[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if T > CONSTRAINTS['t_max']:
                return self._build_obs(), -50.0, True, info
            obj = float(sum(z_vals[i] * final_rhos[i] for i in self.selected))
            reward = obj / 1e6
            info['objective'] = obj
            info['T'] = T
            info['D_vals'] = D_vals
            info['z_vals'] = z_vals
            return self._build_obs(), reward, True, info

        # If action is device pick
        if action < 0 or action >= self.N:
            # invalid action
            self.done = True
            return self._build_obs(), -5.0, True, info

        if action in self.selected:
            # selecting same device twice -> punish
            self.done = True
            return self._build_obs(), -1.0, True, info

        # Add device
        self.selected.append(int(action))

        # If selecting more than allowed, punish and end
        if len(self.selected) > self.S_max:
            self.done = True
            return self._build_obs(), -5.0, True, info

        # Immediate marginal reward: estimate device's raw contribution (rho * z_orig)
        # scaled down so agent can learn numerically
        marginal = self.devices[action]['rho'] * self.devices[action]['z_orig']
        immediate_reward = marginal / 1e6  # small immediate reward

        # If picking this device immediately violates selected-device feasibility, punish and end
        if not feasible_selected_device(self.devices[action], self.D[action], self.z[action]):
            self.done = True
            return self._build_obs(), -20.0, True, info

        # If currently selected devices already violate latency, punish and end
        if not feasible_round_latency(self.selected, self.D, self.devices):
            self.done = True
            return self._build_obs(), -20.0, True, info

        # If haven't hit STOP and haven't filled S_max, episode continues
        # return immediate reward to encourage good picks
        return self._build_obs(), float(immediate_reward), False, info

# -------------------------
# DQN
# -------------------------
if TORCH_OK:
    class ReplayBuffer:
        def __init__(self, capacity=REPLAY_CAPACITY):
            self.buf = deque(maxlen=capacity)
        def push(self, s,a,r,s2,d):
            self.buf.append((s,a,r,s2,d))
        def sample(self, batch_size):
            batch = random.sample(self.buf, batch_size)
            s,a,r,s2,d = map(np.array, zip(*batch))
            return s,a,r,s2,d
        def __len__(self):
            return len(self.buf)

    class QNet(nn.Module):
        def __init__(self, in_dim, out_dim, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim)
            )
        def forward(self, x): return self.net(x)

    class DQNAgent:
        def __init__(self, obs_dim, n_actions, lr=LR):
            self.q = QNet(obs_dim, n_actions)
            self.target = QNet(obs_dim, n_actions)
            self.target.load_state_dict(self.q.state_dict())
            self.opt = optim.Adam(self.q.parameters(), lr=lr)
            self.replay = ReplayBuffer()
            self.eps = EPS_START
            self.n_actions = n_actions
            self.step_count = 0

        def select(self, state):
            if random.random() < self.eps:
                return random.randrange(self.n_actions)
            s = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                qvals = self.q(s).cpu().numpy()[0]
            return int(np.argmax(qvals))

        def store(self, s,a,r,s2,d):
            self.replay.push(s,a,r,s2,d)

        def update(self, batch_size=BATCH_SIZE):
            if len(self.replay) < batch_size:
                return
            s,a,r,s2,d = self.replay.sample(batch_size)
            s = torch.FloatTensor(s)
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r)
            s2 = torch.FloatTensor(s2)
            d = torch.FloatTensor(d)
            qvals = self.q(s).gather(1, a).squeeze()
            with torch.no_grad():
                qnext = self.target(s2).max(1)[0]
                target = r + (1-d) * GAMMA * qnext
            loss = nn.MSELoss()(qvals, target)
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
            self.step_count += 1
            self.eps = max(EPS_MIN, self.eps * EPS_DECAY)
            if self.step_count % 500 == 0:
                self.target.load_state_dict(self.q.state_dict())

# -------------------------
# Training loop
# -------------------------
def train_dqn_on_movielens(devices, episodes=EPISODES):
    env = FederatedEnv(devices)
    obs0 = env.reset()
    obs_dim = obs0.shape[0]
    n_actions = len(devices) + 1  # extra STOP action
    if not TORCH_OK:
        raise RuntimeError("PyTorch not installed. Install torch to train DQN.")
    agent = DQNAgent(obs_dim, n_actions, lr=LR)
    best = {'obj': -1.0, 'info': None, 'selected': None}
    rewards = []
    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action = agent.select(state)
            next_s, r, done, info = env.step(action)
            agent.store(state, action, r, next_s, float(done))
            agent.update(BATCH_SIZE)
            state = next_s
            ep_reward += r
            # if done, break; else continue picking/stop
        rewards.append(ep_reward)
        if info and 'objective' in info:
            obj = info['objective']
            if obj > best['obj']:
                best['obj'] = obj
                best['info'] = info
                best['selected'] = env.selected.copy()
        if (ep+1) % 50 == 0:
            print(f"Episode {ep+1}/{episodes} — ep_reward={ep_reward:.3f} eps={agent.eps:.3f} best_obj={best['obj']:.1f}")
    return agent, best, rewards

# -------------------------
# Random device sizes helper
# -------------------------
def get_random_device_sizes(num_users, num_devices=10, seed=None):
    rng = np.random.default_rng(seed)
    probs = np.ones(num_devices) / num_devices
    sizes = rng.multinomial(num_users, probs)
    return sizes.tolist()

# -------------------------
# Detailed solution print
# -------------------------
def print_solution_details(best, devices):
    if best['info'] is None:
        print("No feasible solution found.")
        return
    selected = best['selected']
    D_vals = best['info']['D_vals']
    z_vals = best['info']['z_vals']
    T = best['info']['T']
    print("\n=== DETAILED SOLUTION STATS ===")
    print(f"Selected devices ({len(selected)}): {selected}")
    print(f"Objective: {best['info']['objective']:.2f}")
    print(f"Round latency T: {T:.6f} s")
    total_users = 0.0
    total_requests = 0.0
    total_energy = 0.0
    for i, dev in enumerate(devices):
        sel = "✅" if i in selected else "❌"
        D = D_vals[i]
        z = z_vals[i]
        rho = dev['rho']
        lat = device_latency(dev, D)
        ene = device_energy(dev, D)
        total_users += D
        total_requests += z
        total_energy += ene
        print(f" Device {i:2d} {sel} | Users_orig={dev['D_orig']:4d} | D_final={D:7.1f} | z_final={z:8.1f} | rho={rho:.4f} | Lat={lat:.6f}s | E={ene:.4f}J")
    print("\nTotals: Users(sum)={:.1f} | Requests(sum)={:.1f} | Energy(sum)={:.4f}J".format(total_users, total_requests, total_energy))
    # check constraints
    lat_ok = T <= CONSTRAINTS['t_max']
    s_ok = (CONSTRAINTS['S_min'] <= len(selected) <= CONSTRAINTS['S_max'])
    print(f"Constraint checks: Latency OK={lat_ok} | Selection count OK={s_ok}")
    if 'off_matrix' in best['info']:
        print("\n=== OFFLOADING MATRIX (users) ===")
        off = best['info']['off_matrix']
        df_off = pd.DataFrame(off, index=[f"Dev{i}" for i in range(len(devices))], columns=[f"Dev{j}" for j in range(len(devices))])
        print(df_off.round(1))


# -------------------------
# Main
# -------------------------
def main():
    print("Loading MovieLens top-1000 movies and creating devices...")
    request_matrix = load_movielens_binary_matrix(MOVIELENS_PATH, top_k_movies=1000)
    num_users = request_matrix.shape[0]
    num_devices = 10
    device_sizes = get_random_device_sizes(num_users, num_devices)
    # print("Random device sizes:", device_sizes, " (sum =", sum(device_sizes), ")")
    devices = create_devices_from_matrix(request_matrix, device_sizes)
    # print("Devices prepared. D_orig list:", [d['D_orig'] for d in devices])
    if not TORCH_OK:
        print("PyTorch not available. Install torch and re-run to train DQN.")
        return
    agent, best, rewards = train_dqn_on_movielens(devices, episodes=EPISODES)
    print("\n=== BEST SOLUTION FOUND ===")
    print_solution_details(best, devices)

if __name__ == "__main__":
    main()

In [6]:
"""
FL device selection (MovieLens) + improved DQN agent for device selection.
- STOP action added (so agent chooses number of devices).
- Softer greedy offloading (partial chunks).
- Immediate marginal reward on selection.
- Prints detailed solution stats at the end.

MODIFIED: removed downsampling in cosine-sim calculation (use all users per device)
and replaced np.random.* calls inside create_devices_from_matrix with local RNG for
reproducible behavior controlled by the `seed` parameter.

MODIFIED AGAIN: Replaced expensive on-the-fly rho recalculation with a faster
heuristic based on a weighted average of the offloaded content's source rho.

MODIFIED AGAIN: Updated final printout to show initial vs. final z and rho values.

MODIFIED AGAIN: Loads ALL movies from the dataset, not just the top 1000.
"""

import math
import random
import time
from collections import deque

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# PyTorch for DQN
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# -------------------------
# Configuration / constants
# -------------------------
MOVIELENS_PATH = r'C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m'  # change if needed

MODEL_SIZE = 5e6      # bits (example)
BANDWIDTH_DN = 50e6   # bps
BANDWIDTH_UP = 10e6   # bps
P_S = 0.5
P_N = 0.3
EPOCHS = 1
# HYPERPARAMETERS DEFINED

CONSTRAINTS = {
    'z_min': 50,
    'z_max': 5000,
    'D_max': 2000,
    't_max': 0.1,
    'e_max': 0.5,
    'S_min': 2,
    'S_max': 9,
    'alpha_n': 0.1
}
# CONSTRAINTS DEFINED

# DQN hyperparameters
EPISODES = 400
BATCH_SIZE = 64
LR = 1e-3
GAMMA = 0.99
EPS_START = 1.0
EPS_MIN = 0.05
EPS_DECAY = 0.995
REPLAY_CAPACITY = 20000
SEED = 1

np.random.seed(SEED)
random.seed(SEED)
if TORCH_OK:
    torch.manual_seed(SEED)

# -------------------------
# Utilities
# -------------------------
def log2(x):
    return math.log(x, 2.0)

# -------------------------
# MovieLens loader + preprocessing
# -------------------------
def load_movielens_binary_matrix(path_dir):
    ratings_path = path_dir.rstrip('/') + '/ratings.dat'
    ratings_col = ['UserID', 'MovieID', 'Rating', 'Timestamp']
    print("Loading ratings from:", ratings_path)
    ratings = pd.read_csv(ratings_path, sep='::', names=ratings_col, engine='python')
    
    # The filtering for top_k_movies has been removed to use all movies
    
    request_matrix = ratings.pivot_table(index='UserID', columns='MovieID', values='Rating', fill_value=0)
    request_matrix = (request_matrix > 0).astype(int)
    print("Request matrix shape (users x movies):", request_matrix.shape)
    return request_matrix

# -------------------------
# Build devices from request matrix
# -------------------------
def create_devices_from_matrix(request_matrix, device_sizes, seed=0):
    """
    Create a list of device dictionaries from the request matrix.
    """
    rng = np.random.default_rng(seed)
    user_ids = request_matrix.index.tolist()
    rng.shuffle(user_ids)
    devices = []
    start = 0
    for i, size in enumerate(device_sizes):
        users = user_ids[start:start+size]
        start += size
        submat = request_matrix.loc[users].values
        D_orig = len(users)
        z_orig = int(submat.sum())
        if D_orig <= 1:
            rho = 1.0
        else:
            Xf = submat.astype(float)
            sims = cosine_similarity(Xf)
            np.fill_diagonal(sims, 0.0)
            total_pairs = Xf.shape[0] * (Xf.shape[0] - 1)
            rho = float(sims.sum() / total_pairs) if total_pairs > 0 else 0.0
        
        f_n = float(rng.uniform(1.0, 2.0) * 1e9)
        beta_n = float(rng.uniform(1e-28, 1e-27))
        C_n = int(rng.integers(5, 10))
        gamma_n = float(rng.uniform(10, 20))
        t_dn = MODEL_SIZE / (BANDWIDTH_DN * log2(1 + gamma_n))
        wn = 2e6
        t_up = wn / (BANDWIDTH_UP * log2(1 + gamma_n))
        t_comp_coef = (EPOCHS * C_n) / f_n
        e_dn = P_S * t_dn
        e_up = P_N * t_up
        e_comp_coef = beta_n * EPOCHS * C_n * (f_n ** 2)
        devices.append({
            'id': i, 'users': users, 'D_orig': D_orig, 'z_orig': z_orig, 'rho': rho,
            'f_n': f_n, 'beta_n': beta_n, 'C_n': C_n, 'gamma_n': gamma_n, 't_dn': t_dn,
            't_up': t_up, 't_comp_coef': t_comp_coef, 'e_dn': e_dn, 'e_up': e_up,
            'e_comp_coef': e_comp_coef, 'wn': wn
        })
    return devices

# -------------------------
# Helpers
# -------------------------
def device_latency(dev, D):
    return dev['t_dn'] + dev['t_comp_coef'] * D + dev['t_up']

def device_energy(dev, D):
    return dev['e_dn'] + dev['e_comp_coef'] * D + dev['e_up']

def feasible_selected_device(dev, D_val, z_val):
    if D_val > CONSTRAINTS['D_max']: return False
    if z_val < CONSTRAINTS['z_min']: return False
    if device_energy(dev, D_val) > CONSTRAINTS['e_max']: return False
    return True

def feasible_round_latency(selected_ids, D_vals, devices):
    if not selected_ids: return False
    latencies = [device_latency(devices[i], D_vals[i]) for i in selected_ids]
    T = max(latencies) if latencies else 0.0
    return T <= CONSTRAINTS['t_max']

def non_selected_request_ok(z_val, T):
    return (z_val + CONSTRAINTS['alpha_n'] * T) <= CONSTRAINTS['z_max']

# -------------------------
# Softer greedy offloading
# -------------------------
def greedy_offloading_soft(devices, selected_idx, D_vals, z_vals):
    if not selected_idx:
        return np.zeros((len(devices), len(devices))), {}

    N = len(devices)
    off_matrix = np.zeros((N, N))
    current_rhos = {i: d['rho'] for i, d in enumerate(devices)}
    per_unit_z = np.zeros(N)
    for i, d in enumerate(devices):
        per_unit_z[i] = d['z_orig'] / max(1.0, d['D_orig'])

    selset = set(selected_idx)
    non_selected = [i for i in range(N) if i not in selset]
    non_selected_sorted = sorted(non_selected, key=lambda x: devices[x]['D_orig'], reverse=True)

    for src in non_selected_sorted:
        remaining = float(D_vals[src])
        if remaining <= 0 or per_unit_z[src] <= 0:
            continue
        
        CHUNK = max(1.0, devices[src]['D_orig'] / 20.0)
        while remaining > 1e-9:
            best_gain = 0.0
            best_j = None
            best_chunk = 0.0
            
            for j in selected_idx:
                if j == src: continue
                
                cap = CONSTRAINTS['D_max'] - D_vals[j]
                if cap <= 0: continue
                
                chunk = min(CHUNK, remaining, cap)
                if chunk <= 0: continue
                
                D_new_j = D_vals[j] + chunk
                z_new_j = z_vals[j] + per_unit_z[src] * chunk

                if not feasible_selected_device(devices[j], D_new_j, z_new_j): continue
                
                D_temp = D_vals.copy()
                D_temp[j] = D_new_j
                if not feasible_round_latency(selected_idx, D_temp, devices): continue

                gain = current_rhos[j] * (per_unit_z[src] * chunk)
                
                if gain > best_gain:
                    best_gain = gain
                    best_j = j
                    best_chunk = chunk
            
            if best_j is None:
                if CHUNK > 1.0:
                    CHUNK = max(1.0, CHUNK / 2.0)
                    continue
                else:
                    break
            
            best_chunk_int = int(round(best_chunk))
            if best_chunk_int <= 0: break

            z_moved = per_unit_z[src] * best_chunk_int
            old_rho_dest = current_rhos[best_j]
            old_z_dest = z_vals[best_j]
            rho_src = current_rhos[src]
            numerator = (old_z_dest * old_rho_dest) + (z_moved * rho_src)
            denominator = old_z_dest + z_moved
            
            if denominator > 0:
                current_rhos[best_j] = numerator / denominator

            D_vals[best_j] += best_chunk_int
            z_vals[best_j] += z_moved
            D_vals[src] = max(0, int(round(D_vals[src] - best_chunk_int)))
            z_vals[src] = max(0.0, z_vals[src] - per_unit_z[src] * best_chunk_int)
            remaining -= best_chunk_int
            off_matrix[src, best_j] += best_chunk_int

    return off_matrix, current_rhos

# -------------------------
# Gym-like environment (with STOP action)
# -------------------------
class FederatedEnv:
    def __init__(self, devices):
        self.devices = devices
        self.N = len(devices)
        self.S_max = CONSTRAINTS['S_max']
        self.S_min = CONSTRAINTS['S_min']
        self.STOP_ACTION = self.N
        self.reset()

    def reset(self):
        self.D = np.array([d['D_orig'] for d in self.devices], dtype=float)
        self.z = np.array([d['z_orig'] for d in self.devices], dtype=float)
        self.selected = []
        self.done = False
        return self._build_obs()

    def _build_obs(self):
        obs = []
        for i, d in enumerate(self.devices):
            Dn = self.D[i] / max(1.0, CONSTRAINTS['D_max'])
            zn = self.z[i] / max(1.0, CONSTRAINTS['z_max'])
            rho = d['rho']
            en = device_energy(d, self.D[i]) / max(1e-9, CONSTRAINTS['e_max'])
            lat = device_latency(d, self.D[i]) / max(1e-9, CONSTRAINTS['t_max'])
            sel = 1.0 if i in self.selected else 0.0
            obs.extend([Dn, zn, rho, en, lat, sel])
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("step after done")
        info = {}
        if action == self.STOP_ACTION:
            if len(self.selected) < self.S_min:
                self.done = True
                return self._build_obs(), -10.0, True, info
            
            self.done = True
            if not all(feasible_selected_device(self.devices[i], self.D[i], self.z[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if not feasible_round_latency(self.selected, self.D, self.devices):
                return self._build_obs(), -50.0, True, info
            
            D_vals = self.D.copy()
            z_vals = self.z.copy()
            off_matrix, final_rhos = greedy_offloading_soft(self.devices, self.selected, D_vals, z_vals)
            
            info['off_matrix'] = off_matrix
            info['final_rhos'] = final_rhos # Store final rhos for printing
            
            T = max(device_latency(self.devices[i], D_vals[i]) for i in self.selected) if self.selected else 0.0
            if not all(non_selected_request_ok(z_vals[i], T) for i in range(self.N) if i not in self.selected):
                return self._build_obs(), -50.0, True, info
            if not all(feasible_selected_device(self.devices[i], D_vals[i], z_vals[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if T > CONSTRAINTS['t_max']:
                return self._build_obs(), -50.0, True, info
            
            obj = float(sum(z_vals[i] * final_rhos[i] for i in self.selected))
            reward = obj / 1e6
            info['objective'] = obj
            info['T'] = T
            info['D_vals'] = D_vals
            info['z_vals'] = z_vals
            return self._build_obs(), reward, True, info

        if action < 0 or action >= self.N:
            self.done = True
            return self._build_obs(), -5.0, True, info
        if action in self.selected:
            self.done = True
            return self._build_obs(), -1.0, True, info

        self.selected.append(int(action))
        if len(self.selected) > self.S_max:
            self.done = True
            return self._build_obs(), -5.0, True, info

        marginal = self.devices[action]['rho'] * self.devices[action]['z_orig']
        immediate_reward = marginal / 1e6

        if not feasible_selected_device(self.devices[action], self.D[action], self.z[action]):
            self.done = True
            return self._build_obs(), -20.0, True, info
        if not feasible_round_latency(self.selected, self.D, self.devices):
            self.done = True
            return self._build_obs(), -20.0, True, info

        return self._build_obs(), float(immediate_reward), False, info

# -------------------------
# DQN
# -------------------------
if TORCH_OK:
    class ReplayBuffer:
        def __init__(self, capacity=REPLAY_CAPACITY):
            self.buf = deque(maxlen=capacity)
        def push(self, s,a,r,s2,d):
            self.buf.append((s,a,r,s2,d))
        def sample(self, batch_size):
            batch = random.sample(self.buf, batch_size)
            s,a,r,s2,d = map(np.array, zip(*batch))
            return s,a,r,s2,d
        def __len__(self):
            return len(self.buf)

    class QNet(nn.Module):
        def __init__(self, in_dim, out_dim, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim)
            )
        def forward(self, x): return self.net(x)

    class DQNAgent:
        def __init__(self, obs_dim, n_actions, lr=LR):
            self.q = QNet(obs_dim, n_actions)
            self.target = QNet(obs_dim, n_actions)
            self.target.load_state_dict(self.q.state_dict())
            self.opt = optim.Adam(self.q.parameters(), lr=lr)
            self.replay = ReplayBuffer()
            self.eps = EPS_START
            self.n_actions = n_actions
            self.step_count = 0

        def select(self, state, already_selected): # Add 'already_selected'
            if random.random() < self.eps:
                # Epsilon-greedy: pick a random *valid* action
                possible_actions = [i for i in range(self.n_actions) if i not in already_selected]
                # Ensure the STOP action is always possible (if not already picked, which it can't be)
                if self.n_actions - 1 not in already_selected: # self.n_actions-1 is the STOP action
                    possible_actions.append(self.n_actions - 1)
                
                # This check is needed in case all devices are selected
                if not possible_actions: 
                    return self.n_actions - 1 # Just STOP
                
                return random.choice(possible_actions)
        
            # Greedy:
            s = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                qvals = self.q(s).cpu().numpy()[0]

            # --- THIS IS THE FIX ---
            # Mask all actions that are already in 'already_selected'
            if already_selected:
                qvals[already_selected] = -np.inf 
            # --- END OF FIX ---
            
            return int(np.argmax(qvals))

        def store(self, s,a,r,s2,d):
            self.replay.push(s,a,r,s2,d)

        def update(self, batch_size=BATCH_SIZE):
            if len(self.replay) < batch_size: return
            s,a,r,s2,d = self.replay.sample(batch_size)
            s = torch.FloatTensor(s)
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r)
            s2 = torch.FloatTensor(s2)
            d = torch.FloatTensor(d)
            qvals = self.q(s).gather(1, a).squeeze()
            with torch.no_grad():
                qnext = self.target(s2).max(1)[0]
                target = r + (1-d) * GAMMA * qnext
            loss = nn.MSELoss()(qvals, target)
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
            self.step_count += 1
            self.eps = max(EPS_MIN, self.eps * EPS_DECAY)
            if self.step_count % 500 == 0:
                self.target.load_state_dict(self.q.state_dict())

# -------------------------
# Training loop
# -------------------------
def train_dqn_on_movielens(devices, episodes=EPISODES):
    """
    Trains the DQN agent on the FederatedEnv.
    """
    env = FederatedEnv(devices)
    obs0 = env.reset()
    obs_dim = obs0.shape[0]
    # N device actions + 1 STOP action
    n_actions = len(devices) + 1 
    
    if not TORCH_OK:
        raise RuntimeError("PyTorch not installed. Install torch to train DQN.")
        
    agent = DQNAgent(obs_dim, n_actions, lr=LR)
    best = {'obj': -1.0, 'info': None, 'selected': None}
    rewards = []
    
    print(f"Starting DQN training for {episodes} episodes...")
    
    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_reward = 0.0
        
        while not done:
            # --- MODIFICATION ---
            # Pass the env.selected list to the agent so it can mask
            # already-chosen actions. This is crucial for efficiency.
            action = agent.select(state, env.selected) 
            
            next_s, r, done, info = env.step(action)
            
            # Store the experience in the replay buffer
            agent.store(state, action, r, next_s, float(done))
            
            # Perform one step of optimization on the policy network
            agent.update(BATCH_SIZE)
            
            state = next_s
            ep_reward += r
            
        rewards.append(ep_reward)
        
        # Check if this episode produced a new best solution
        if info and 'objective' in info:
            obj = info['objective']
            if obj > best['obj']:
                best['obj'] = obj
                best['info'] = info
                best['selected'] = env.selected.copy()
                
        if (ep+1) % 50 == 0:
            avg_rew = np.mean(rewards[-50:]) # Moving average reward
            print(f"Episode {ep+1}/{episodes} — AvgRew(50)={avg_rew:.3f} eps={agent.eps:.3f} best_obj={best['obj']:.1f}")
            
    return agent, best, rewards

# -------------------------
# Random device sizes helper
# -------------------------
def get_random_device_sizes(num_users, num_devices=10, seed=None):
    rng = np.random.default_rng(seed)
    probs = np.ones(num_devices) / num_devices
    sizes = rng.multinomial(num_users, probs)
    return sizes.tolist()

# -------------------------
# Detailed solution print
# -------------------------
def print_solution_details(best, devices):
    if best['info'] is None:
        print("No feasible solution found.")
        return
    
    selected = best['selected']
    D_vals = best['info']['D_vals']
    z_vals = best['info']['z_vals']
    T = best['info']['T']
    final_rhos = best['info'].get('final_rhos', {}) # Safely get the final rhos

    print("\n=== DETAILED SOLUTION STATS ===")
    print(f"Selected devices ({len(selected)}): {selected}")
    print(f"Objective: {best['info']['objective']:.2f}")
    print(f"Round latency T: {T:.6f} s")
    
    # --- DETAILED HEADER ---
    print("-" * 120)
    header = (
        f"{'Device':<10s} | {'D_orig':>7s} | {'D_final':>8s} | "
        f"{'z_initial':>9s} | {'z_final':>9s} | "
        f"{'rho_initial':>11s} | {'rho_final':>11s} | "
        f"{'Latency':>10s} | {'Energy':>8s}"
    )
    print(header)
    print("-" * 120)

    total_users = 0.0
    total_requests = 0.0
    total_energy = 0.0

    for i, dev in enumerate(devices):
        sel_char = "✅" if i in selected else "❌"
        
        # Initial values from device creation
        d_orig = dev['D_orig']
        z_initial = dev['z_orig']
        rho_initial = dev['rho']
        
        # Final values after offloading
        d_final = D_vals[i]
        z_final = z_vals[i]
        # Use final rho if available, otherwise it's the initial rho
        rho_final = final_rhos.get(i, rho_initial)
        
        # Final calculated latency and energy
        lat = device_latency(dev, d_final)
        ene = device_energy(dev, d_final)
        
        total_users += d_final
        total_requests += z_final
        total_energy += ene

        # --- DETAILED PRINT ROW ---
        row = (
            f"Dev {i:<2d} {sel_char:<2s} | {d_orig:>7d} | {d_final:>8.1f} | "
            f"{z_initial:>9d} | {z_final:>9.1f} | "
            f"{rho_initial:>11.4f} | {rho_final:>11.4f} | "
            f"{lat:>10.6f}s | {ene:>8.4f}J"
        )
        print(row)
        
    print("-" * 120)
    print("\nTotals: Users(sum)={:.1f} | Requests(sum)={:.1f} | Energy(sum)={:.4f}J".format(total_users, total_requests, total_energy))
    
    lat_ok = T <= CONSTRAINTS['t_max']
    s_ok = (CONSTRAINTS['S_min'] <= len(selected) <= CONSTRAINTS['S_max'])
    print(f"Constraint checks: Latency OK={lat_ok} | Selection count OK={s_ok}")
    
    if 'off_matrix' in best['info']:
        print("\n=== OFFLOADING MATRIX (users) ===")
        off = best['info']['off_matrix']
        df_off = pd.DataFrame(off, index=[f"Dev{i}" for i in range(len(devices))], columns=[f"Dev{j}" for j in range(len(devices))])
        print(df_off.round(1))

# -------------------------
# Main
# -------------------------
def main():
    print("Loading MovieLens all movies and creating devices...")
    request_matrix = load_movielens_binary_matrix(MOVIELENS_PATH)
    num_users = request_matrix.shape[0]
    num_devices = 10
    device_sizes = get_random_device_sizes(num_users, num_devices)
    devices = create_devices_from_matrix(request_matrix, device_sizes)
    
    if not TORCH_OK:
        print("PyTorch not available. Install torch and re-run to train DQN.")
        return
        
    agent, best, rewards = train_dqn_on_movielens(devices, episodes=EPISODES)
    print("\n=== BEST SOLUTION FOUND ===")
    print_solution_details(best, devices)

if __name__ == "__main__":
    main()

Loading MovieLens all movies and creating devices...
Loading ratings from: C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m/ratings.dat
Request matrix shape (users x movies): (6040, 3706)
Starting DQN training for 400 episodes...
Episode 50/400 — AvgRew(50)=-15.391 eps=0.369 best_obj=125957.8
Episode 100/400 — AvgRew(50)=-3.212 eps=0.050 best_obj=125957.8
Episode 150/400 — AvgRew(50)=-1.796 eps=0.050 best_obj=125957.8
Episode 200/400 — AvgRew(50)=-0.588 eps=0.050 best_obj=125957.8
Episode 250/400 — AvgRew(50)=-2.004 eps=0.050 best_obj=125957.8
Episode 300/400 — AvgRew(50)=-1.285 eps=0.050 best_obj=125957.8
Episode 350/400 — AvgRew(50)=-1.981 eps=0.050 best_obj=125957.8
Episode 400/400 — AvgRew(50)=-1.979 eps=0.050 best_obj=125957.8

=== BEST SOLUTION FOUND ===

=== DETAILED SOLUTION STATS ===
Selected devices (6): [9, 4, 3, 1, 8, 6]
Objective: 125957.81
Round latency T: 0.083825 s
---------------------------------------------------------------------------------------

In [12]:
# --- Imports (no changes) ---
import math
import random
import time
from collections import deque

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# PyTorch for DQN
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# --- Configuration / constants (no changes) ---
MOVIELENS_PATH = r'C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m'  # change if needed
MODEL_SIZE = 5e6      # bits (example)
BANDWIDTH_DN = 50e6   # bps
BANDWIDTH_UP = 10e6   # bps
P_S = 0.5
P_N = 0.3
EPOCHS = 1

CONSTRAINTS = {
    'z_min': 50,
    'z_max': 5000,
    'D_max': 2000,
    't_max': 0.1,
    'e_max': 0.5,
    'S_min': 2,
    'S_max': 9,
    'alpha_n': 0.1
}

# DQN hyperparameters
EPISODES = 400
BATCH_SIZE = 64
LR = 1e-3
GAMMA = 0.99
EPS_START = 1.0
EPS_MIN = 0.05
EPS_DECAY = 0.995
REPLAY_CAPACITY = 20000
SEED = 1

# --- NEW: Autoencoder Hyperparameters ---
LATENT_DIM = 64
AE_EPOCHS = 20
AE_BATCH_SIZE = 128
AE_LR = 1e-3

np.random.seed(SEED)
random.seed(SEED)
if TORCH_OK:
    torch.manual_seed(SEED)

# --- Utilities (no changes) ---
def log2(x):
    return math.log(x, 2.0)

# --- MovieLens loader (no changes) ---
def load_movielens_binary_matrix(path_dir):
    ratings_path = path_dir.rstrip('/') + '/ratings.dat'
    ratings_col = ['UserID', 'MovieID', 'Rating', 'Timestamp']
    print("Loading ratings from:", ratings_path)
    ratings = pd.read_csv(ratings_path, sep='::', names=ratings_col, engine='python', encoding='latin-1')
    
    request_matrix = ratings.pivot_table(index='UserID', columns='MovieID', values='Rating', fill_value=0)
    request_matrix = (request_matrix > 0).astype(int)
    print("Request matrix shape (users x movies):", request_matrix.shape)
    return request_matrix

# -------------------------
# NEW: Autoencoder Definition
# -------------------------
if TORCH_OK:
    class Autoencoder(nn.Module):
        def __init__(self, input_dim, latent_dim=LATENT_DIM):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, latent_dim)
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 256),
                nn.ReLU(),
                nn.Linear(256, input_dim),
                nn.Sigmoid() # Use Sigmoid for [0, 1] output, matching our binary matrix
            )

        def forward(self, x):
            latent = self.encoder(x)
            reconstructed = self.decoder(latent)
            return reconstructed

# -------------------------
# NEW: Autoencoder Training Function
# -------------------------
def train_autoencoder(data_matrix, device):
    """
    Trains the autoencoder on the full request matrix.
    """
    if not TORCH_OK:
        raise RuntimeError("PyTorch not found")
        
    num_users, num_movies = data_matrix.shape
    
    # Convert numpy matrix to PyTorch tensors and create DataLoader
    # We use .astype(float) because the model expects floats, not ints
    tensor_data = torch.tensor(data_matrix.values.astype(np.float32)).to(device)
    dataset = TensorDataset(tensor_data, tensor_data) # Input and target are the same
    dataloader = DataLoader(dataset, batch_size=AE_BATCH_SIZE, shuffle=True)
    
    model = Autoencoder(input_dim=num_movies, latent_dim=LATENT_DIM).to(device)
    criterion = nn.MSELoss() # Mean Squared Error is a good choice
    optimizer = optim.Adam(model.parameters(), lr=AE_LR)
    
    print(f"\n--- Training Autoencoder for {AE_EPOCHS} epochs... ---")
    model.train()
    for epoch in range(AE_EPOCHS):
        total_loss = 0
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        if (epoch + 1) % 5 == 0:
            print(f"AE Epoch [{epoch+1}/{AE_EPOCHS}], Loss: {avg_loss:.6f}")
            
    print("--- Autoencoder training complete. ---")
    return model

# -------------------------
# NEW: Latent Vector Generation
# -------------------------
def get_latent_vectors(model, data_matrix, device):
    """
    Passes the full dataset through the encoder to get latent vectors.
    """
    print("Generating latent vectors for all users...")
    model.eval()
    with torch.no_grad():
        tensor_data = torch.tensor(data_matrix.values.astype(np.float32)).to(device)
        latent_vectors = model.encoder(tensor_data)
    return latent_vectors.cpu().numpy()


# -------------------------
# MODIFIED: Build devices from request matrix
# -------------------------
def create_devices_from_matrix(
    request_matrix, 
    device_sizes, 
    latent_user_vectors,  # <-- NEW
    user_id_to_row_map,   # <-- NEW
    seed=0
):
    """
    Create a list of device dictionaries from the request matrix.
    MODIFIED to use latent vectors for rho calculation.
    """
    rng = np.random.default_rng(seed)
    user_ids = request_matrix.index.tolist()
    rng.shuffle(user_ids)
    
    devices = []
    start = 0
    
    for i, size in enumerate(device_sizes):
        if size <= 0:
            continue
            
        users = user_ids[start:start+size]
        start += size
        
        # Get original data stats (z, D) - this logic is unchanged
        submat = request_matrix.loc[users].values
        D_orig = len(users)
        z_orig = int(submat.sum())
        
        if D_orig <= 1:
            rho = 1.0
        else:
            # --- START OF MODIFIED RHO CALCULATION ---
            # 1. Get the row indices for these users
            row_indices = [user_id_to_row_map[uid] for uid in users]
            
            # 2. Get the corresponding latent vectors
            device_latent_vecs = latent_user_vectors[row_indices]
            
            # 3. Calculate similarity on the *latent vectors* [~600, 64]
            sims = cosine_similarity(device_latent_vecs)
            # --- END OF MODIFIED RHO CALCULATION ---
            
            # This part is the same as before
            np.fill_diagonal(sims, 0.0)
            total_pairs = D_orig * (D_orig - 1)
            rho = float(sims.sum() / total_pairs) if total_pairs > 0 else 0.0
        
        # Device hardware stats (unchanged)
        f_n = float(rng.uniform(1.0, 2.0) * 1e9)
        beta_n = float(rng.uniform(1e-28, 1e-27))
        C_n = int(rng.integers(5, 10))
        gamma_n = float(rng.uniform(10, 20))
        t_dn = MODEL_SIZE / (BANDWIDTH_DN * log2(1 + gamma_n))
        wn = 2e6
        t_up = wn / (BANDWIDTH_UP * log2(1 + gamma_n))
        t_comp_coef = (EPOCHS * C_n) / f_n
        e_dn = P_S * t_dn
        e_up = P_N * t_up
        e_comp_coef = beta_n * EPOCHS * C_n * (f_n ** 2)
        
        devices.append({
            'id': i, 'users': users, 'D_orig': D_orig, 'z_orig': z_orig, 'rho': rho,
            'f_n': f_n, 'beta_n': beta_n, 'C_n': C_n, 'gamma_n': gamma_n, 't_dn': t_dn,
            't_up': t_up, 't_comp_coef': t_comp_coef, 'e_dn': e_dn, 'e_up': e_up,
            'e_comp_coef': e_comp_coef, 'wn': wn
        })
    return devices

# --- Helpers (no changes) ---
def device_latency(dev, D):
    return dev['t_dn'] + dev['t_comp_coef'] * D + dev['t_up']

def device_energy(dev, D):
    return dev['e_dn'] + dev['e_comp_coef'] * D + dev['e_up']

def feasible_selected_device(dev, D_val, z_val):
    if D_val > CONSTRAINTS['D_max']: return False
    if z_val < CONSTRAINTS['z_min']: return False
    if device_energy(dev, D_val) > CONSTRAINTS['e_max']: return False
    return True

def feasible_round_latency(selected_ids, D_vals, devices):
    if not selected_ids: return False
    latencies = [device_latency(devices[i], D_vals[i]) for i in selected_ids]
    T = max(latencies) if latencies else 0.0
    return T <= CONSTRAINTS['t_max']

def non_selected_request_ok(z_val, T):
    return (z_val + CONSTRAINTS['alpha_n'] * T) <= CONSTRAINTS['z_max']

# --- Greedy offloading (no changes) ---
def greedy_offloading_soft(devices, selected_idx, D_vals, z_vals):
    # This function remains identical. It relies on the
    # initial 'rho' and 'z' values, which we have now
    # calculated more effectively.
    if not selected_idx:
        return np.zeros((len(devices), len(devices))), {}

    N = len(devices)
    off_matrix = np.zeros((N, N))
    current_rhos = {i: d['rho'] for i, d in enumerate(devices)}
    per_unit_z = np.zeros(N)
    for i, d in enumerate(devices):
        per_unit_z[i] = d['z_orig'] / max(1.0, d['D_orig'])

    selset = set(selected_idx)
    non_selected = [i for i in range(N) if i not in selset]
    non_selected_sorted = sorted(non_selected, key=lambda x: devices[x]['D_orig'], reverse=True)

    for src in non_selected_sorted:
        remaining = float(D_vals[src])
        if remaining <= 0 or per_unit_z[src] <= 0:
            continue
        
        CHUNK = max(1.0, devices[src]['D_orig'] / 20.0)
        while remaining > 1e-9:
            best_gain = 0.0
            best_j = None
            best_chunk = 0.0
            
            for j in selected_idx:
                if j == src: continue
                
                cap = CONSTRAINTS['D_max'] - D_vals[j]
                if cap <= 0: continue
                
                chunk = min(CHUNK, remaining, cap)
                if chunk <= 0: continue
                
                D_new_j = D_vals[j] + chunk
                z_new_j = z_vals[j] + per_unit_z[src] * chunk

                if not feasible_selected_device(devices[j], D_new_j, z_new_j): continue
                
                D_temp = D_vals.copy()
                D_temp[j] = D_new_j
                if not feasible_round_latency(selected_idx, D_temp, devices): continue

                gain = current_rhos[j] * (per_unit_z[src] * chunk)
                
                if gain > best_gain:
                    best_gain = gain
                    best_j = j
                    best_chunk = chunk
            
            if best_j is None:
                if CHUNK > 1.0:
                    CHUNK = max(1.0, CHUNK / 2.0)
                    continue
                else:
                    break
            
            best_chunk_int = int(round(best_chunk))
            if best_chunk_int <= 0: break

            z_moved = per_unit_z[src] * best_chunk_int
            old_rho_dest = current_rhos[best_j]
            old_z_dest = z_vals[best_j]
            rho_src = current_rhos[src]
            numerator = (old_z_dest * old_rho_dest) + (z_moved * rho_src)
            denominator = old_z_dest + z_moved
            
            if denominator > 0:
                current_rhos[best_j] = numerator / denominator

            D_vals[best_j] += best_chunk_int
            z_vals[best_j] += z_moved
            D_vals[src] = max(0, int(round(D_vals[src] - best_chunk_int)))
            z_vals[src] = max(0.0, z_vals[src] - per_unit_z[src] * best_chunk_int)
            remaining -= best_chunk_int
            off_matrix[src, best_j] += best_chunk_int

    return off_matrix, current_rhos

# --- Gym environment (no changes) ---
class FederatedEnv:
    def __init__(self, devices):
        self.devices = devices
        self.N = len(devices)
        self.S_max = CONSTRAINTS['S_max']
        self.S_min = CONSTRAINTS['S_min']
        self.STOP_ACTION = self.N
        self.reset()

    def reset(self):
        self.D = np.array([d['D_orig'] for d in self.devices], dtype=float)
        self.z = np.array([d['z_orig'] for d in self.devices], dtype=float)
        self.selected = []
        self.done = False
        return self._build_obs()

    def _build_obs(self):
        obs = []
        for i, d in enumerate(self.devices):
            Dn = self.D[i] / max(1.0, CONSTRAINTS['D_max'])
            zn = self.z[i] / max(1.0, CONSTRAINTS['z_max'])
            rho = d['rho']
            en = device_energy(d, self.D[i]) / max(1e-9, CONSTRAINTS['e_max'])
            lat = device_latency(d, self.D[i]) / max(1e-9, CONSTRAINTS['t_max'])
            sel = 1.0 if i in self.selected else 0.0
            obs.extend([Dn, zn, rho, en, lat, sel])
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("step after done")
        info = {}
        if action == self.STOP_ACTION:
            if len(self.selected) < self.S_min:
                self.done = True
                return self._build_obs(), -10.0, True, info
            
            self.done = True
            if not all(feasible_selected_device(self.devices[i], self.D[i], self.z[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if not feasible_round_latency(self.selected, self.D, self.devices):
                return self._build_obs(), -50.0, True, info
            
            D_vals = self.D.copy()
            z_vals = self.z.copy()
            off_matrix, final_rhos = greedy_offloading_soft(self.devices, self.selected, D_vals, z_vals)
            
            info['off_matrix'] = off_matrix
            info['final_rhos'] = final_rhos
            
            T = max(device_latency(self.devices[i], D_vals[i]) for i in self.selected) if self.selected else 0.0
            if not all(non_selected_request_ok(z_vals[i], T) for i in range(self.N) if i not in self.selected):
                return self._build_obs(), -50.0, True, info
            if not all(feasible_selected_device(self.devices[i], D_vals[i], z_vals[i]) for i in self.selected):
                return self._build_obs(), -50.0, True, info
            if T > CONSTRAINTS['t_max']:
                return self._build_obs(), -50.0, True, info
            
            obj = float(sum(z_vals[i] * final_rhos[i] for i in self.selected))
            reward = obj / 1e6
            info['objective'] = obj
            info['T'] = T
            info['D_vals'] = D_vals
            info['z_vals'] = z_vals
            return self._build_obs(), reward, True, info

        if action < 0 or action >= self.N:
            self.done = True
            return self._build_obs(), -5.0, True, info
        if action in self.selected:
            self.done = True
            return self._build_obs(), -1.0, True, info

        self.selected.append(int(action))
        if len(self.selected) > self.S_max:
            self.done = True
            return self._build_obs(), -5.0, True, info

        marginal = self.devices[action]['rho'] * self.devices[action]['z_orig']
        immediate_reward = marginal / 1e6

        if not feasible_selected_device(self.devices[action], self.D[action], self.z[action]):
            self.done = True
            return self._build_obs(), -20.0, True, info
        if not feasible_round_latency(self.selected, self.D, self.devices):
            self.done = True
            return self._build_obs(), -20.0, True, info

        return self._build_obs(), float(immediate_reward), False, info

# --- DQN (ReplayBuffer, QNet) (no changes) ---
if TORCH_OK:
    class ReplayBuffer:
        def __init__(self, capacity=REPLAY_CAPACITY):
            self.buf = deque(maxlen=capacity)
        def push(self, s,a,r,s2,d):
            self.buf.append((s,a,r,s2,d))
        def sample(self, batch_size):
            batch = random.sample(self.buf, batch_size)
            s,a,r,s2,d = map(np.array, zip(*batch))
            return s,a,r,s2,d
        def __len__(self):
            return len(self.buf)

    class QNet(nn.Module):
        def __init__(self, in_dim, out_dim, hidden=256):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, out_dim)
            )
        def forward(self, x): return self.net(x)

# --- DQN (DQNAgent - MODIFIED for action masking) ---
    class DQNAgent:
        def __init__(self, obs_dim, n_actions, lr=LR):
            self.q = QNet(obs_dim, n_actions)
            self.target = QNet(obs_dim, n_actions)
            self.target.load_state_dict(self.q.state_dict())
            self.opt = optim.Adam(self.q.parameters(), lr=lr)
            self.replay = ReplayBuffer()
            self.eps = EPS_START
            self.n_actions = n_actions # This now includes STOP action
            self.step_count = 0

        def select(self, state, already_selected): # <-- MODIFIED
            if random.random() < self.eps:
                # Epsilon-greedy: pick a random *valid* action
                possible_actions = [i for i in range(self.N) if i not in already_selected]
                possible_actions.append(self.STOP_ACTION) # STOP is always possible
                return random.choice(possible_actions)
            
            # Greedy:
            s = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                qvals = self.q(s).cpu().numpy()[0]

            # --- ACTION MASKING FIX ---
            if already_selected:
                qvals[already_selected] = -np.inf # Mask already-picked devices
            # --- END OF FIX ---
            
            return int(np.argmax(qvals))

        def store(self, s,a,r,s2,d):
            self.replay.push(s,a,r,s2,d)

        def update(self, batch_size=BATCH_SIZE):
            if len(self.replay) < batch_size: return
            s,a,r,s2,d = self.replay.sample(batch_size)
            s = torch.FloatTensor(s)
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r)
            s2 = torch.FloatTensor(s2)
            d = torch.FloatTensor(d)
            qvals = self.q(s).gather(1, a).squeeze()
            with torch.no_grad():
                qnext = self.target(s2).max(1)[0]
                target = r + (1-d) * GAMMA * qnext
            loss = nn.MSELoss()(qvals, target)
            self.opt.zero_grad()
            loss.backward()
            self.opt.step()
            self.step_count += 1
            self.eps = max(EPS_MIN, self.eps * EPS_DECAY)
            if self.step_count % 500 == 0:
                self.target.load_state_dict(self.q.state_dict())

# --- Training loop (MODIFIED for action masking) ---
def train_dqn_on_movielens(devices, episodes=EPISODES):
    env = FederatedEnv(devices)
    obs0 = env.reset()
    obs_dim = obs0.shape[0]
    n_actions = len(devices) + 1 # N devices + 1 STOP action
    
    if not TORCH_OK:
        raise RuntimeError("PyTorch not installed. Install torch to train DQN.")
        
    agent = DQNAgent(obs_dim, n_actions, lr=LR)
    
    # --- Pass N and STOP_ACTION to agent for masking ---
    agent.N = len(devices)
    agent.STOP_ACTION = agent.N
    # ---
    
    best = {'obj': -1.0, 'info': None, 'selected': None}
    rewards = []
    
    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            # --- MODIFIED ---
            # Pass the env.selected list to the agent for masking
            action = agent.select(state, env.selected) 
            
            next_s, r, done, info = env.step(action)
            agent.store(state, action, r, next_s, float(done))
            agent.update(BATCH_SIZE)
            state = next_s
            ep_reward += r
            
        rewards.append(ep_reward)
        if info and 'objective' in info:
            obj = info['objective']
            if obj > best['obj']:
                best['obj'] = obj
                best['info'] = info
                best['selected'] = env.selected.copy()
                
        if (ep+1) % 50 == 0:
            avg_rew = np.mean(rewards[-50:])
            print(f"Episode {ep+1}/{episodes} — AvgRew(50)={avg_rew:.3f} eps={agent.eps:.3f} best_obj={best['obj']:.1f}")
            
    return agent, best, rewards

def reconstruct_device_datasets(request_matrix, devices, best_solution):
    """
    "Moves the samples" based on the offloading matrix to create the final
    datasets for the selected devices.
    """
    print("\n--- Phase 2: Reconstructing Datasets for Training ---")
    if not best_solution['info']:
        print("No feasible solution found, cannot reconstruct datasets.")
        return {}

    selected_ids = best_solution['selected']
    off_matrix = best_solution['info']['off_matrix']
    
    final_device_datasets = {}

    for device_id in selected_ids:
        # Start with the device's original users
        original_users = set(devices[device_id]['users'])
        
        # Find users offloaded TO this device
        offloaded_users_to_here = set()
        
        # The offloading matrix is [from, to], so we look at the column for this device
        for src_device_id in range(len(devices)):
            if src_device_id != device_id:
                # Get the User IDs from the source device's user list
                num_users_to_move = int(off_matrix[src_device_id, device_id])
                if num_users_to_move > 0:
                    # For this simulation, we'll just take the first N users
                    # In a real system, you'd have a specific list of who was moved
                    users_to_move = devices[src_device_id]['users'][:num_users_to_move]
                    offloaded_users_to_here.update(users_to_move)

        final_user_set = list(original_users.union(offloaded_users_to_here))
        
        # Create the new data shard (a pandas DataFrame)
        new_dataset = request_matrix.loc[final_user_set]
        final_device_datasets[device_id] = new_dataset
        
        print(f"Device {device_id}: Original users={len(original_users)}, "
              f"Offloaded-in={len(offloaded_users_to_here)}, "
              f"Final dataset size={len(new_dataset)}")
              
    return final_device_datasets

def simulate_fl_training_round(final_datasets):
    """
    Simulates one round of FL training on the reconstructed datasets.
    """
    print("\n--- Simulating FL Training Round ---")
    if not final_datasets:
        print("No datasets to train on.")
        return

    # In a real FL system, you would:
    # 1. Send a global model to each selected device.
    # 2. Train a local model on each device using its `final_datasets[device_id]`.
    # 3. Collect the model updates (e.g., weights).
    # 4. Aggregate them (e.g., FedAvg) to update the global model.

    print("Training on selected devices:")
    for device_id, dataset in final_datasets.items():
        num_users, num_movies = dataset.shape
        num_samples = dataset.values.sum()
        print(f" -> Device {device_id}: Training model on {num_users} users, "
              f"{num_samples} total samples.")
    
    print("\nGlobal model updated successfully.")

# --- Random device sizes helper (no changes) ---
def get_random_device_sizes(num_users, num_devices=10, seed=None):
    rng = np.random.default_rng(seed)
    probs = np.ones(num_devices) / num_devices
    sizes = rng.multinomial(num_users, probs)
    return sizes.tolist()

# --- Detailed solution print (no changes) ---
def print_solution_details(best, devices):
    if best['info'] is None:
        print("No feasible solution found.")
        return
    
    selected = best['selected']
    D_vals = best['info']['D_vals']
    z_vals = best['info']['z_vals']
    T = best['info']['T']
    final_rhos = best['info'].get('final_rhos', {})

    print("\n=== DETAILED SOLUTION STATS =====")
    print(f"Selected devices ({len(selected)}): {selected}")
    print(f"Objective: {best['info']['objective']:.2f}")
    print(f"Round latency T: {T:.6f} s")
    
    print("-" * 120)
    header = (
        f"{'Device':<10s} | {'D_orig':>7s} | {'D_final':>8s} | "
        f"{'z_initial':>9s} | {'z_final':>9s} | "
        f"{'rho_initial':>11s} | {'rho_final':>11s} | "
        f"{'Latency':>10s} | {'Energy':>8s}"
    )
    print(header)
    print("-" * 120)

    total_users = 0.0
    total_requests = 0.0
    total_energy = 0.0

    for i, dev in enumerate(devices):
        sel_char = "✅" if i in selected else "❌"
        d_orig = dev['D_orig']
        z_initial = dev['z_orig']
        rho_initial = dev['rho']
        d_final = D_vals[i]
        z_final = z_vals[i]
        rho_final = final_rhos.get(i, rho_initial)
        lat = device_latency(dev, d_final)
        ene = device_energy(dev, d_final)
        
        total_users += d_final
        total_requests += z_final
        total_energy += ene

        row = (
            f"Dev {i:<2d} {sel_char:<2s} | {d_orig:>7d} | {d_final:>8.1f} | "
            f"{z_initial:>9d} | {z_final:>9.1f} | "
            f"{rho_initial:>11.4f} | {rho_final:>11.4f} | "
            f"{lat:>10.6f}s | {ene:>8.4f}J"
        )
        print(row)
        
    print("-" * 120)
    print("\nTotals: Users(sum)={:.1f} | Requests(sum)={:.1f} | Energy(sum)={:.4f}J".format(total_users, total_requests, total_energy))
    
    lat_ok = T <= CONSTRAINTS['t_max']
    s_ok = (CONSTRAINTS['S_min'] <= len(selected) <= CONSTRAINTS['S_max'])
    print(f"Constraint checks: Latency OK={lat_ok} | Selection count OK={s_ok}")
    
    if 'off_matrix' in best['info']:
        print("\n=== OFFLOADING MATRIX (users) ===")
        off = best['info']['off_matrix']
        df_off = pd.DataFrame(off, index=[f"Dev{i}" for i in range(len(devices))], columns=[f"Dev{j}" for j in range(len(devices))])
        print(df_off.round(1))

# -----------------------------------------------------
# --- CODE BLOCK FOR BASELINES & PHASE 2 (PATH A) ---
# -----------------------------------------------------

def evaluate_selection(selected, devices):
    """
    A helper function that evaluates a given team of selected devices.
    
    This function replicates the final evaluation logic from the
    FederatedEnv's "STOP" action.
    """
    N = len(devices)
    D_vals = np.array([d['D_orig'] for d in devices], dtype=float)
    z_vals = np.array([d['z_orig'] for d in devices], dtype=float)
    
    # 1. Pre-check constraints (before offloading)
    if not selected:
        return {'obj': -1.0, 'info': None, 'selected': selected}
        
    if not all(feasible_selected_device(devices[i], D_vals[i], z_vals[i]) for i in selected):
        print("Baseline failed: Pre-check device constraints")
        return {'obj': -1.0, 'info': None, 'selected': selected}
        
    if not feasible_round_latency(selected, D_vals, devices):
        print("Baseline failed: Pre-check round latency")
        return {'obj': -1.0, 'info': None, 'selected': selected}

    # 2. Run greedy offloading
    off_matrix, final_rhos = greedy_offloading_soft(devices, selected, D_vals, z_vals)
    
    # 3. Post-check constraints (after offloading)
    T = max(device_latency(devices[i], D_vals[i]) for i in selected) if selected else 0.0
    
    if T > CONSTRAINTS['t_max']:
        print(f"Baseline failed: Post-check latency > t_max (T={T:.4f})")
        return {'obj': -1.0, 'info': None, 'selected': selected}
        
    if not all(non_selected_request_ok(z_vals[i], T) for i in range(N) if i not in selected):
        print("Baseline failed: Post-check non-selected request")
        return {'obj': -1.0, 'info': None, 'selected': selected}
        
    if not all(feasible_selected_device(devices[i], D_vals[i], z_vals[i]) for i in selected):
        print("Baseline failed: Post-check selected device constraints")
        return {'obj': -1.0, 'info': None, 'selected': selected}

    # 4. All constraints passed! Calculate final objective.
    obj = float(sum(z_vals[i] * final_rhos.get(i, devices[i]['rho']) for i in selected))
    
    # 5. Build the same 'best' dictionary as the DQN
    info = {
        'objective': obj,
        'T': T,
        'D_vals': D_vals,
        'z_vals': z_vals,
        'off_matrix': off_matrix,
        'final_rhos': final_rhos
    }
    best = {'obj': obj, 'info': info, 'selected': selected}
    
    return best

def baseline_greedy_rho(devices, S_target):
    """
    Baseline 1: Selects the top S_target devices with the highest initial rho.
    """
    print(f"\n--- Running Baseline: Greedy-by-Rho (S_target={S_target}) ---")
    if S_target <= 0:
        return {'obj': -1.0, 'info': None, 'selected': []}
        
    # Sort device IDs by their initial rho
    sorted_ids = sorted(range(len(devices)), 
                        key=lambda i: devices[i]['rho'], 
                        reverse=True)
    
    # Select the top S_target devices
    selected = sorted_ids[:S_target]
    
    # Evaluate this selection
    return evaluate_selection(selected, devices)

def baseline_random(devices, S_target, seed=0):
    """
    Baseline 2: Selects S_target devices at random.
    """
    print(f"\n--- Running Baseline: Random (S_target={S_target}) ---")
    if S_target <= 0:
        return {'obj': -1.0, 'info': None, 'selected': []}
        
    rng = np.random.default_rng(seed)
    device_ids = list(range(len(devices)))
    rng.shuffle(device_ids)
    
    # Select the first S_target devices from the shuffled list
    selected = device_ids[:S_target]
    
    # Evaluate this selection
    return evaluate_selection(selected, devices)

def reconstruct_device_datasets(request_matrix, devices, best_solution):
    """
    "Moves the samples" based on the offloading matrix to create the final
    datasets for the selected devices.
    """
    print("\n--- Phase 2: Reconstructing Datasets for Training ---")
    if not best_solution or not best_solution['info']:
        print("No feasible solution found, cannot reconstruct datasets.")
        return {}

    selected_ids = best_solution['selected']
    off_matrix = best_solution['info']['off_matrix']
    
    final_device_datasets = {}

    for device_id in selected_ids:
        # Start with the device's original users
        original_users = set(devices[device_id]['users'])
        
        # Find users offloaded TO this device
        offloaded_users_to_here = set()
        
        # The offloading matrix is [from, to], so we look at the column for this device
        for src_device_id in range(len(devices)):
            if src_device_id != device_id:
                # Get the User IDs from the source device's user list
                num_users_to_move = int(off_matrix[src_device_id, device_id])
                if num_users_to_move > 0:
                    # For this simulation, we'll just take the first N users
                    # from the source device's original list
                    users_to_move = devices[src_device_id]['users'][:num_users_to_move]
                    offloaded_users_to_here.update(users_to_move)

        final_user_set = list(original_users.union(offloaded_users_to_here))
        
        # Create the new data shard (a pandas DataFrame)
        # Ensure we only select users that actually exist in the matrix
        valid_users = [u for u in final_user_set if u in request_matrix.index]
        if not valid_users:
            print(f"Warning: Device {device_id} ended with no valid users.")
            continue
            
        new_dataset = request_matrix.loc[valid_users]
        final_device_datasets[device_id] = new_dataset
        
        print(f"Device {device_id}: Original users={len(original_users)}, "
              f"Offloaded-in={len(offloaded_users_to_here)}, "
              f"Final dataset size={len(new_dataset)}")
              
    return final_device_datasets

def simulate_fl_training_round(final_datasets):
    """
    Simulates one round of FL training on the reconstructed datasets.
    """
    print("\n--- Simulating FL Training Round ---")
    if not final_datasets:
        print("No datasets to train on.")
        return

    print("Training on selected devices:")
    for device_id, dataset in final_datasets.items():
        num_users, num_movies = dataset.shape
        num_samples = dataset.values.sum()
        print(f" -> Device {device_id}: Training model on {num_users} users, "
              f"{num_samples} total samples.")
    
    print("\nGlobal model updated successfully.")
# -------------------------
# Main
# -------------------------
def main():
    if not TORCH_OK:
        print("PyTorch not available. Install torch and re-run.")
        return
        
    # Set device for PyTorch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- Step 1: Load Data ---
    print("Loading MovieLens all movies...")
    request_matrix = load_movielens_binary_matrix(MOVIELENS_PATH)
    
    # --- Step 2: Train Autoencoder ---
    ae_model = train_autoencoder(request_matrix, device)
    
    # --- Step 3: Get Latent Vectors ---
    latent_user_vectors = get_latent_vectors(ae_model, request_matrix, device)
    
    # Create a map from UserID to its row index in the latent matrix
    user_id_to_row_map = {user_id: i for i, user_id in enumerate(request_matrix.index)}
    
    # --- Step 4: Create Devices (using latent vectors) ---
    print("\nCreating devices using AE-derived rho...")
    num_users = request_matrix.shape[0]
    num_devices = 10
    device_sizes = get_random_device_sizes(num_users, num_devices, seed=SEED)
    
    devices = create_devices_from_matrix(
        request_matrix, 
        device_sizes, 
        latent_user_vectors,  # <-- Pass AE data
        user_id_to_row_map,   # <-- Pass AE data
        seed=SEED
    )
    
    print(f"Devices prepared. Initial rho values: {[round(d['rho'], 3) for d in devices]}")
    
    # =============================================================
    #           PHASE 1: RUN ALL SELECTION ALGORITHMS
    # =============================================================

    # --- ALGORITHM 1: DQN (Your main algorithm) ---
    print("\n\n=== RUNNING ALGORITHM 1: DQN AGENT ===")
    agent, best_dqn, rewards = train_dqn_on_movielens(devices, episodes=EPISODES)
    print("\n=== BEST DQN SOLUTION FOUND ===")
    print_solution_details(best_dqn, devices)

    # For a fair comparison, we tell the baselines to
    # select the SAME NUMBER of devices as the DQN did.
    if best_dqn and best_dqn.get('selected'):
        S_target = len(best_dqn['selected'])
    else:
        # Fallback if DQN found no solution
        S_target = (CONSTRAINTS['S_min'] + CONSTRAINTS['S_max']) // 2 
        print(f"\nWarning: DQN found no solution. Using fallback S_target={S_target} for baselines.")

    # --- ALGORITHM 2: GREEDY-BY-RHO (Baseline 1) ---
    best_greedy = baseline_greedy_rho(devices, S_target)
    print("\n=== GREEDY-BY-RHO SOLUTION FOUND ===")
    print_solution_details(best_greedy, devices)

    # --- ALGORITHM 3: RANDOM (Baseline 2) ---
    best_random = baseline_random(devices, S_target, seed=SEED)
    print("\n=== RANDOM SOLUTION FOUND ===")
    print_solution_details(best_random, devices)

    # =============================================================
    #           FINAL COMPARISON SUMMARY
    # =============================================================
    print("\n\n" + "="*40)
    print("     FINAL RESULTS SUMMARY (PHASE 1)")
    print("="*40)
    print(f"Target number of devices: {S_target}")
    print(f"DQN Agent Objective:     {best_dqn.get('obj', -1.0):.2f}")
    print(f"Greedy-by-Rho Objective: {best_greedy.get('obj', -1.0):.2f}")
    print(f"Random Objective:        {best_random.get('obj', -1.0):.2f}")
    print("="*40)
    
    # =============================================================
    #           PHASE 2: EXECUTE THE ROUND
    # =============================================================
    
    # We proceed to Phase 2 using the *best* solution found (the DQN's)
    # This simulates what you would do in a real system.
    
    # Step 6: "Move the samples"
    final_datasets = reconstruct_device_datasets(request_matrix, devices, best_dqn)
    
    # Step 7: "Then train"
    simulate_fl_training_round(final_datasets)

if __name__ == "__main__":
    main()

Using device: cpu
Loading MovieLens all movies...
Loading ratings from: C:\Users\darsh\OneDrive\Desktop\2nd YEAR\Summer\Project stuff\ml-1m/ratings.dat
Request matrix shape (users x movies): (6040, 3706)

--- Training Autoencoder for 20 epochs... ---
AE Epoch [5/20], Loss: 0.044692
AE Epoch [10/20], Loss: 0.035164
AE Epoch [15/20], Loss: 0.028999
AE Epoch [20/20], Loss: 0.026858
--- Autoencoder training complete. ---
Generating latent vectors for all users...

Creating devices using AE-derived rho...
Devices prepared. Initial rho values: [0.693, 0.693, 0.687, 0.687, 0.681, 0.677, 0.688, 0.679, 0.687, 0.682]


=== RUNNING ALGORITHM 1: DQN AGENT ===
Episode 50/400 — AvgRew(50)=-6.936 eps=0.194 best_obj=685325.4
Episode 100/400 — AvgRew(50)=-0.578 eps=0.050 best_obj=685325.4
Episode 150/400 — AvgRew(50)=0.839 eps=0.050 best_obj=685325.4
Episode 200/400 — AvgRew(50)=-1.758 eps=0.050 best_obj=685325.4
Episode 250/400 — AvgRew(50)=-1.034 eps=0.050 best_obj=685325.4
Episode 300/400 — AvgRew(5